# 1. 데이터 불러오기

이 섹션의 목표는 원본 데이터를 분석할 수 있는 표 하나로 정리하는 것입니다.

1. 데이터 폴더 경로를 잡고, 폴더별 파일 개수를 확인합니다.
2. JSON 하나를 열어 실제 형식이 대회 명세와 같은지 확인합니다.
3. 모든 JSON을 읽어 bbox 1개 = 1행인 표(`ann_df`)를 만듭니다.
4. 이미지 파일과 어노테이션의 짝이 맞는지 확인합니다.

데이터 오류를 고치는 작업은 하지 않습니다. 여기서는 어떤 상태인지 세어 보기만 하고, 수정은 이후 섹션(오류 탐지)에서 합니다.

In [ ]:
import hashlib
import json
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw

# 데이터 폴더 경로 (노트북 위치 기준 상대경로)
DATA_DIR = Path("../data/raw/sprint_ai_project1_data")
TRAIN_IMG_DIR = DATA_DIR / "train_images"
TRAIN_ANN_DIR = DATA_DIR / "train_annotations"
TEST_IMG_DIR = DATA_DIR / "test_images"

# 경로가 틀리면 아래 셀이 전부 0개로 나오므로, 여기서 바로 멈추게 합니다.
assert DATA_DIR.exists(), f"데이터 폴더를 찾을 수 없습니다: {DATA_DIR} (노트북이 notebooks/ 폴더 안에 있는지 확인하세요)"
print("데이터 폴더:", DATA_DIR)

## 1-1. 대회 제공 데이터 설명

아래 내용은 대회 페이지의 설명을 그대로 옮긴 것입니다. 실제 데이터가 이 설명과 같은지는 이후 단계에서 확인합니다.

### 파일 구성

| 폴더 | 설명 |
|---|---|
| `train_images` | 훈련 데이터(이미지) |
| `train_annotations` | 훈련 데이터(annotation) |
| `test_images` | 테스트 데이터(이미지) |

### JSON 항목

필수 여부의 M, O, C는 명세에 따로 설명이 없습니다. 보통 M은 필수, O는 선택, C는 조건부를 뜻합니다.

#### images: 이미지 정보

| 항목명 | 타입 | 필수 여부 | 설명 |
|---|---|---|---|
| `images` | Object | M | 약제 이미지정보 |
| `images[].id` | Number | M | 약제 이미지식별자 |
| `images[].width` | Number | M | 약제 이미지너비 |
| `images[].height` | Number | M | 약제 이미지높이 |
| `images[].file_name` | String | M | 약제 이미지파일명 |
| `images[].drug_N` | String | M | 알약코드 |
| `images[].drug_S` | String | M | 알약상태 |
| `images[].back_color` | String | M | 촬영배경 |
| `images[].drug_dir` | String | M | 알약방향 |
| `images[].light_color` | String | M | 촬영조명 |
| `images[].camera_la` | Number | M | 카메라위도 |
| `images[].camera_lo` | Number | M | 카메라경도 |
| `images[].size` | Number | M | 이미지 사이즈 |
| `images[].dl_idx` | String | M | 알약 식별자 |
| `images[].dl_mapping_code` | String | M | 제품코드 |
| `images[].dl_name` | String | M | 제품명 |
| `images[].dl_name_en` | String | O | 제품명(영어) |
| `images[].img_key` | String | M | 이미지 링크 |
| `images[].dl_material` | String | M | 성분명 |
| `images[].dl_material_en` | String | O | 성분명(영어) |
| `images[].dl_custom_shape` | String | M | 제조 모양 |
| `images[].dl_company` | String | M | 제조사명 |
| `images[].dl_company_en` | String | O | 제조사명(영어) |
| `images[].di_company_mf` | String | M | 위탁제조사명 |
| `images[].di_company_mf_en` | String | O | 위탁제조사명(영어) |
| `images[].item_seq` | Number | M | 품목기준코드 |
| `images[].di_item_permit_date` | Date | O | 허가일자 |
| `images[].di_class_no` | String | M | 약품 분류 |
| `images[].di_etc_otc_code` | String | M | 전문의약품/일반의약품 |
| `images[].di_edi_code` | String | M | EDI 코드 |
| `images[].chart` | String | M | 알약 제형 |
| `images[].drug_shape` | String | M | 알약 모양 |
| `images[].thick` | Number | M | 알약 두께 |
| `images[].leng_long` | Number | M | 알약 장축 |
| `images[].leng_short` | Number | M | 알약 단축 |
| `images[].print_front` | String | C | 식별문자_앞 |
| `images[].print_back` | String | C | 식별문자_뒤 |
| `images[].color_class1` | String | M | 색상 1 |
| `images[].color_class2` | String | O | 색상 2 |
| `images[].line_front` | String | M | 알약 앞면분할선 여부 |
| `images[].line_back` | String | M | 알약 뒷면분할선 여부 |
| `images[].img_regist_ts` | Date | O | 약학정보원 이미지 생성일 |
| `images[].form_code_name` | String | M | 정제 분류명 |
| `images[].mark_code_front_anal` | String | M | 알약 앞면마크 형태 |
| `images[].mark_code_back_anal` | String | M | 알약 뒷면마크 형태 |
| `images[].mark_code_front_img` | String | M | 알약 앞면마크 이미지 |
| `images[].mark_code_back_img` | String | M | 알약 뒷면마크 이미지 |
| `images[].mark_code_front` | String | M | 알약 앞면마크 코드 |
| `images[].mark_code_back` | String | M | 알약 뒷면마크 코드 |
| `images[].change_date` | Date | O | 변경일자 |

#### annotations: 라벨링 정보

| 항목명 | 타입 | 필수 여부 | 설명 |
|---|---|---|---|
| `annotations` | Object | M | 라벨링정보 |
| `annotations[].area` | Number | M | 바운딩박스 면적 |
| `annotations[].iscrowd` | Number | O | 평가 분류 |
| `annotations[].bbox` | List | M | bbox 좌표 |
| `annotations[].category_id` | Number | M | category 아이디 |
| `annotations[].ignore` | Number | O | 무시 여부 |
| `annotations[].segmentation` | List | O | 라벨링바운딩 박스 |
| `annotations[].image_id` | Number | M | 이미지 아이디 |
| `annotations[].id` | Number | M | 어노테이션 아이디 |

#### categories, type

| 항목명 | 타입 | 필수 여부 | 설명 |
|---|---|---|---|
| `categories` | Object | M | 라이선스 |
| `categories[].supercategory` | String | M | 슈퍼 카테고리 |
| `categories[].id` | Number | M | 카테고리 아이디 |
| `categories[].name` | String | C | 카테고리 명 |
| `type` | type | O | json 타입 |

## 1-2. 데이터 폴더 구조와 파일 개수

데이터 폴더를 세 가지 방법으로 확인합니다.

1. 트리: 폴더가 어떻게 생겼는지 봅니다. 폴더 이름 옆의 숫자는 `[바로 아래 폴더 수 · 그 아래 전체 파일 수]`입니다. 한 폴더 안에서는 앞의 3개 항목만 보여 주고, 나머지는 개수로 요약합니다.
2. 표: 최상위 폴더별로 확장자마다 파일이 몇 개인지 셉니다. png, json이 아닌 파일이 섞여 있으면 여기서 드러납니다.
3. 그래프: 표의 폴더별 합계를 막대로 그립니다.

마지막 셀에서는 이후 분석에 쓸 파일 목록을 만듭니다. `rglob`은 하위 폴더가 몇 단계로 들어 있어도 파일을 모두 찾습니다. 여기서 센 개수가 위 표의 png, json 개수와 같아야 합니다.

In [ ]:
def folder_summary(folder):
    """폴더 옆에 붙일 요약: [바로 아래 폴더 수 · 하위 전체 파일 수]"""
    n_dirs = sum(1 for p in folder.iterdir() if p.is_dir())
    n_files = sum(1 for p in folder.rglob("*") if p.is_file())
    return f"[폴더 {n_dirs}개 · 파일 {n_files}개]"


def show_tree(folder, max_depth=4, max_items=3, prefix=""):
    """폴더 구조를 트리 모양으로 출력한다.

    max_depth: 몇 단계 아래까지 펼칠지
    max_items: 한 폴더 안에서 이름을 보여 줄 항목 수 (나머지는 개수로 요약)
    """
    children = sorted(folder.iterdir())
    dirs = [p for p in children if p.is_dir()]
    files = [p for p in children if p.is_file()]

    # 폴더를 먼저, 파일을 나중에 보여 준다.
    shown = (dirs + files)[:max_items]
    hidden_dirs = len(dirs) - sum(p.is_dir() for p in shown)
    hidden_files = len(files) - sum(p.is_file() for p in shown)
    has_hidden = hidden_dirs > 0 or hidden_files > 0

    for i, child in enumerate(shown):
        is_last = (i == len(shown) - 1) and not has_hidden
        branch = "└── " if is_last else "├── "

        if child.is_dir():
            print(f"{prefix}{branch}{child.name}/  {folder_summary(child)}")
            if max_depth > 1:
                # 마지막 항목 아래는 세로줄 없이 들여쓴다.
                next_prefix = prefix + ("    " if is_last else "│   ")
                show_tree(child, max_depth - 1, max_items, next_prefix)
        else:
            print(f"{prefix}{branch}{child.name}")

    if has_hidden:
        rest = []
        if hidden_dirs:
            rest.append(f"폴더 {hidden_dirs}개")
        if hidden_files:
            rest.append(f"파일 {hidden_files}개")
        print(f"{prefix}└── ... 외 {', '.join(rest)}")


print(f"{DATA_DIR.name}/  {folder_summary(DATA_DIR)}")
show_tree(DATA_DIR)

In [ ]:
# 최상위 폴더마다 하위의 모든 파일을 확장자별로 셉니다.
records = []
for top in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    for p in top.rglob("*"):
        if p.is_file():
            records.append({"folder": top.name, "ext": p.suffix.lower() or "(확장자 없음)"})
file_df = pd.DataFrame(records)

# margins=True: 마지막 행과 열에 합계를 붙입니다.
count_df = pd.crosstab(file_df["folder"], file_df["ext"], margins=True, margins_name="합계")
count_df

In [ ]:
# 폴더별 전체 파일 수를 막대그래프로 그립니다. (마지막 '합계' 행은 제외)
totals = count_df["합계"].drop("합계").sort_values()

fig, ax = plt.subplots(figsize=(7, 2.5))
bars = ax.barh(totals.index, totals.values, height=0.5, color="#2a78d6")

# 막대 끝에 개수를 적습니다.
ax.bar_label(bars, padding=4, color="#52514e")

# 그래프 안 글자는 영어로 둡니다. matplotlib 기본 글꼴에는 한글이 없어서 한글이 네모로 깨집니다.
ax.set_title("Files per folder (all subfolders included)", loc="left")
ax.set_xlabel("number of files", color="#898781")
ax.set_xlim(0, totals.max() * 1.15)  # 막대 끝 숫자가 잘리지 않도록 여유를 둡니다.
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
train_img_paths = sorted(TRAIN_IMG_DIR.rglob("*.png"))
test_img_paths = sorted(TEST_IMG_DIR.rglob("*.png"))
ann_paths = sorted(TRAIN_ANN_DIR.rglob("*.json"))

print(f"train 이미지 : {len(train_img_paths)}장")
print(f"test 이미지  : {len(test_img_paths)}장")
print(f"train JSON   : {len(ann_paths)}개")

## 1-3. JSON 읽기

모든 JSON을 한 번씩 읽어 `loaded` 리스트에 `(경로, 내용)` 형태로 담아 둡니다. 이후 셀은 파일을 다시 열지 않고 이 리스트를 씁니다.

- Windows에서 인코딩을 지정하지 않고 열면 기본값(cp949)으로 읽어 한글 때문에 오류가 납니다. 그래서 UTF-8로 엽니다.
- 읽다가 오류가 난 파일은 멈추지 않고 `failed_paths`에 모아 둡니다.

In [ ]:
def read_json(path):
    """JSON 파일 하나를 읽어 dict로 돌려준다. 읽지 못하면 None을 돌려준다."""
    try:
        # utf-8-sig: 파일 맨 앞에 BOM이 있어도, 없어도 읽을 수 있는 UTF-8 방식
        with open(path, encoding="utf-8-sig") as f:
            return json.load(f)
    except (OSError, UnicodeDecodeError, json.JSONDecodeError) as e:
        print(f"[읽기 실패] {path.name}: {e}")
        return None


loaded = []        # (경로, JSON 내용) 목록
failed_paths = []  # 읽지 못한 파일 목록

for path in ann_paths:
    data = read_json(path)
    if data is None:
        failed_paths.append(path)
    else:
        loaded.append((path, data))

print(f"읽기 성공: {len(loaded)}개 / 실패: {len(failed_paths)}개")

## 1-4. JSON 하나 열어 보기

대회 명세에 따르면 JSON은 `images`, `annotations`, `categories`와 선택 항목인 `type`으로 되어 있습니다.
첫 번째 파일을 열어 실제 형식이 명세와 같은지 눈으로 확인합니다.

In [ ]:
sample_path, sample = loaded[0]

print("파일:", sample_path.relative_to(TRAIN_ANN_DIR))
print("최상위 키:", list(sample.keys()))
print()

# annotations, categories는 항목이 적어서 전체를 출력합니다.
# ensure_ascii=False: 한글을 \uXXXX가 아닌 글자 그대로 출력
print("[annotations]")
print(json.dumps(sample.get("annotations"), ensure_ascii=False, indent=2))
print("[categories]")
print(json.dumps(sample.get("categories"), ensure_ascii=False, indent=2))

# images는 항목이 많아서, 이번 분석에 쓰는 항목만 출력합니다.
print("[images[0] 주요 항목]")
sample_img = (sample.get("images") or [{}])[0]  # images가 비어 있어도 오류가 나지 않게
for key in ["file_name", "width", "height", "dl_idx", "dl_name"]:
    print(f"  {key}: {sample_img.get(key)}")

## 1-5. 전체 JSON 구조 확인

파일 개수를 보면 "JSON 1개 = 알약 1개" 구조로 추정됩니다(이미지 1장당 JSON 평균 3.3개).
모든 JSON이 정말 "이미지 1개 + 어노테이션 1개"인지, `(images 개수, annotations 개수)` 조합별로 파일 수를 세어 확인합니다.

In [ ]:
structure = Counter(
    (len(data.get("images", [])), len(data.get("annotations", [])))
    for _, data in loaded
)

for (n_img, n_ann), count in sorted(structure.items()):
    print(f"images {n_img}개 + annotations {n_ann}개 → JSON {count}개")

## 1-6. 어노테이션 표 만들기

모든 JSON을 bbox 1개 = 1행인 표 `ann_df`로 정리합니다. 이후 EDA 섹션은 모두 이 표를 사용합니다.

| 열 | 출처 | 설명 |
|---|---|---|
| `json_path` | 파일 경로 | 문제가 있는 행의 원본 파일을 찾을 때 사용 |
| `file_name`, `img_width`, `img_height` | `images[0]` | 이미지 파일명과 크기 |
| `category_id` | `annotations[]` | 클래스 번호 (제출 파일에 쓰는 값) |
| `dl_idx`, `dl_name` | `images[0]` | 알약 식별자와 약 이름 |
| `bbox_x`, `bbox_y`, `bbox_w`, `bbox_h` | `annotations[].bbox` | 좌상단 x, y와 너비, 높이 (픽셀) |
| `bbox_ok` | 계산 | bbox가 명세의 유효 기준(값 4개짜리 리스트)을 만족하는지 |

명세상 `category_id`는 `dl_idx` 값으로 만들었다고 되어 있습니다. 둘이 실제로 같은지는 오류 탐지 섹션에서 비교하려고 두 열을 모두 남겨 둡니다.

In [ ]:
def to_rows(path, data):
    """JSON 하나를 표의 행 목록으로 바꾼다. 어노테이션 1개가 1행이 된다."""
    images = data.get("images", [])
    img = images[0] if images else {}  # 명세상 JSON 하나에 이미지 정보는 1개

    rows = []
    for ann in data.get("annotations", []):
        bbox = ann.get("bbox")
        # 명세의 유효 기준: 리스트이고 길이가 정확히 4 ([x, y, w, h])
        bbox_ok = isinstance(bbox, list) and len(bbox) == 4
        x, y, w, h = bbox if bbox_ok else (None, None, None, None)

        rows.append({
            "json_path": path.relative_to(TRAIN_ANN_DIR).as_posix(),
            "file_name": img.get("file_name"),
            "img_width": img.get("width"),
            "img_height": img.get("height"),
            "category_id": ann.get("category_id"),
            "dl_idx": img.get("dl_idx"),
            "dl_name": img.get("dl_name"),
            "bbox_x": x,
            "bbox_y": y,
            "bbox_w": w,
            "bbox_h": h,
            "bbox_ok": bbox_ok,
        })
    return rows


rows = []
for path, data in loaded:
    rows.extend(to_rows(path, data))
ann_df = pd.DataFrame(rows)

print(f"표 크기: {ann_df.shape[0]}행 x {ann_df.shape[1]}열")
print(f"bbox 유효: {ann_df['bbox_ok'].sum()}개 / 유효하지 않음: {(~ann_df['bbox_ok']).sum()}개")
ann_df.head()

## 1-7. 이미지와 어노테이션 짝 확인

이미지 파일명과 JSON 안의 `file_name`을 비교합니다.

- 어노테이션이 없는 이미지: 학습에 쓰면 모델이 그 안의 알약을 "배경"으로 배웁니다.
- 이미지 파일이 없는 어노테이션: 학습에 쓸 수 없는 라벨입니다.

In [ ]:
train_names = {p.name for p in train_img_paths}
ann_names = set(ann_df["file_name"].dropna())

# 하위 폴더에 같은 이름의 이미지가 있으면 파일명만으로는 구분이 안 되므로 미리 확인합니다.
if len(train_names) != len(train_img_paths):
    print(f"[주의] 파일명이 겹치는 이미지가 있습니다: {len(train_img_paths) - len(train_names)}개")

no_ann_images = sorted(train_names - ann_names)  # 이미지는 있는데 어노테이션이 없음
no_image_anns = sorted(ann_names - train_names)  # 어노테이션은 있는데 이미지 파일이 없음

print(f"어노테이션이 있는 이미지      : {len(train_names & ann_names)}장 / 전체 {len(train_names)}장")
print(f"어노테이션이 없는 이미지      : {len(no_ann_images)}장")
print(f"이미지 파일이 없는 어노테이션 : {len(no_image_anns)}개 (파일명 기준)")

# 해당하는 항목이 있으면 앞의 5개만 보여 줍니다.
if no_ann_images:
    print("  예시(어노테이션 없음):", no_ann_images[:5])
if no_image_anns:
    print("  예시(이미지 없음):", no_image_anns[:5])

## 1-8. 정리

- 파일 구성: train 이미지 232장, test 이미지 842장, train JSON 763개입니다. 트리, 표, rglob 개수가 모두 같고, png와 json 외의 파일은 없습니다.
- 폴더 구조: `train_annotations/조합 폴더(114개)/알약 코드 폴더/이미지별 JSON`입니다. 조합 폴더 이름과 이미지 파일명 앞부분에 그 사진에 들어 있는 알약 코드가 적혀 있습니다.
- JSON 형식: 대회 명세와 같습니다(`images`, `type`, `annotations`, `categories`). 763개 모두 문제없이 읽혔습니다.
- JSON 구조: 763개 전부 "이미지 1개 + 어노테이션 1개"입니다. 즉 JSON 1개가 알약 1개입니다.
- bbox: 763개 모두 값 4개짜리로 형식은 유효합니다. 값이 이미지 안에 있는지는 아직 확인하지 않았습니다.
- 짝 확인: 232장 모두 어노테이션이 있고, 이미지 파일이 없는 어노테이션도 없습니다.
- 이미지당 어노테이션은 평균 3.3개입니다. 파일명의 알약 코드 수와 비교하면 라벨이 빠진 알약을 찾을 수 있습니다.

다음 섹션부터는 `ann_df` 표를 사용합니다.

# 2. 샘플 이미지 보기

train, test 이미지에서 각각 6장을 무작위로 골라 원본 그대로 봅니다. bbox는 그리지 않고, 사진 자체가 어떻게 생겼는지(배경, 조명, 알약 개수와 배치)를 확인하는 용도입니다.

- 무작위 추출에 seed를 고정해서, 다시 실행해도 팀원이 실행해도 같은 사진이 뽑힙니다. 다른 사진을 보고 싶으면 `seed` 값을 바꿉니다.
- 사진 위에는 파일명, 크기(가로x세로), 색 모드를 적습니다. 색 모드는 `RGB`(3채널), `RGBA`(투명도 포함 4채널), `L`(흑백) 등이 있으며, 모델에 넣기 전에 채널 수를 맞춰야 하는지 판단하는 데 씁니다.
- 그래프 안 글자는 1-2와 같은 이유로 영어로 둡니다.

In [ ]:
def show_samples(img_paths, title, n=6, seed=42, ncols=3):
    """이미지 목록에서 n장을 무작위로 골라 격자로 보여 준다.

    seed가 같으면 항상 같은 사진이 뽑힌다.
    random.Random(seed)는 이 함수 안에서만 쓰는 난수 생성기라서, 노트북의 다른 난수에 영향을 주지 않는다.
    """
    picked = random.Random(seed).sample(img_paths, n)
    nrows = (n + ncols - 1) // ncols  # 올림 나눗셈: 6장을 3열로 놓으면 2행

    # 원본 사진이 세로로 긴 편(976x1280)이라 칸도 세로로 길게 잡습니다.
    # layout="constrained": 전체 제목과 사진별 제목이 겹치지 않게 자동으로 간격을 맞춥니다.
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 5.5 * nrows), layout="constrained")

    # 사진 칸의 눈금을 모두 끕니다. 칸이 사진 수보다 많으면 남는 칸은 빈칸으로 보입니다.
    for ax in axes.flat:
        ax.axis("off")

    for ax, path in zip(axes.flat, picked):
        # with 문: 이미지를 다 쓰면 파일을 바로 닫습니다.
        with Image.open(path) as img:
            ax.imshow(img)
            ax.set_title(f"{path.name}\n{img.width}x{img.height}, {img.mode}", fontsize=8)

    fig.suptitle(f"{title}: {n} random samples (seed={seed})", x=0.01, ha="left")
    plt.show()

## 2-1. train 이미지

In [ ]:
show_samples(train_img_paths, "train_images")

## 2-2. test 이미지

In [ ]:
show_samples(test_img_paths, "test_images")

## 2-3. 정리

seed=42로 뽑은 train 6장, test 6장을 본 결과입니다.

- 크기와 색 모드: 12장 모두 976x1280, RGB입니다. 전체 이미지가 모두 같은지는 따로 확인이 필요합니다.
- 배경과 조명: 배경은 모두 연한 청회색 계열이고, 사진마다 밝기와 색감이 조금씩 다릅니다. 일부 사진은 모서리가 밝게 번져 원형 테두리가 보입니다.
- 알약 개수와 배치: 사진마다 알약이 3개 또는 4개이고, 서로 떨어져 놓여 있습니다. 겹치거나 가장자리에 걸친 알약은 없었습니다.
- 파일명 앞부분의 코드: `K-` 뒤의 6자리 숫자는 알약 종류를 나타내는 코드로 추정합니다. 근거는 다음과 같습니다.
    - 같은 알약이 보이는 사진들의 파일명에는 같은 코드가 들어 있었습니다.
    - 파일명에 있는 코드의 종류 수가 사진에 보이는 알약 수와 같았습니다.
    - 근거의 범위: 위 샘플과 별도로, `train_images` 폴더에서 파일명과 이미지를 대조하며 눈으로 살펴본 결과입니다.
- train과 test: 배경, 조명, 알약 크기가 비슷해 보입니다. 같은 알약(주황색 원형 "HT")이 여러 사진에 반복해서 나옵니다.
- 노이즈: test `1449.png` 가운데에 먼지로 보이는 작은 점이 있습니다.

# 3. 중복 이미지

train, test 이미지 1074장 중에 완전히 같은 이미지가 있는지 확인합니다. 중복을 찾기만 하고 지우지는 않습니다.

- 같은 이미지의 기준: 크기, 색 모드, 모든 픽셀 값이 같은 이미지입니다. 파일 내용(바이트)이 아니라 픽셀로 비교하므로, 같은 사진을 다른 이름이나 다른 압축 방식으로 다시 저장한 경우도 찾습니다.
- 살짝 다르게 찍은 사진은 증강 목적으로 넣었을 수 있어서 이번에는 찾지 않습니다.

## 3-1. 이미지 해시값 계산

이미지마다 MD5 해시 함수로 해시값을 계산합니다. 해시 함수에는 이미지의 색 모드, 크기, 압축을 푼 픽셀 값 전체를 넣습니다. 픽셀이 하나라도 다르면 해시값이 사실상 항상 달라지므로, 해시값이 같은 이미지를 같은 이미지로 봅니다.

1074장을 모두 열기 때문에 실행에 시간이 조금 걸립니다.

### 용어 설명

- 해시 함수: 어떤 데이터를 넣든 정해진 길이의 값을 돌려주는 함수입니다. 같은 데이터를 넣으면 항상 같은 값이 나오고, 데이터가 조금만 달라도 전혀 다른 값이 나옵니다.
- 해시값: 해시 함수가 돌려준 값입니다. 데이터 전체를 비교하는 대신 이 값만 비교해서 두 데이터가 같은지 판단할 수 있습니다.
- MD5: 해시 함수의 한 종류입니다. 입력 크기와 상관없이 128비트, 즉 16진수 32글자의 해시값을 만듭니다. 예를 들어 빈 데이터의 MD5 해시값은 `d41d8cd98f00b204e9800998ecf8427e`입니다.

In [ ]:
def pixel_hash(path):
    """이미지의 색 모드, 크기, 픽셀 값으로 MD5 해시값을 계산한다."""
    with Image.open(path) as img:
        h = hashlib.md5()  # MD5 해시 계산기를 만든다.
        # update: 해시 계산기에 데이터를 넣는다. 여러 번 넣으면 넣은 데이터를 이어 붙인 전체의 해시값이 된다.
        # 크기와 색 모드도 함께 넣어서, 픽셀 바이트가 우연히 같아도 크기나 색 모드가 다르면 다른 이미지로 본다.
        h.update(f"{img.mode}_{img.width}x{img.height}".encode())
        h.update(img.tobytes())  # 압축을 푼 픽셀 값 전체
        return h.hexdigest()     # 해시값을 16진수 32글자 문자열로 돌려준다.


hash_records = []
for split, paths in [("train", train_img_paths), ("test", test_img_paths)]:
    for path in paths:
        hash_records.append({"split": split, "file_name": path.name, "hash": pixel_hash(path)})
hash_df = pd.DataFrame(hash_records)

print(f"해시값을 계산한 이미지 : {len(hash_df)}장")
print(f"서로 다른 해시값       : {hash_df['hash'].nunique()}개")

## 3-2. 중복 묶음 개수

해시값이 같은 이미지끼리 하나의 묶음으로 보고, 묶음을 세 종류로 나눠 셉니다.

- train 안: train 이미지끼리 같은 경우
- test 안: test 이미지끼리 같은 경우
- train-test 사이: 묶음 안에 train과 test 이미지가 함께 있는 경우

In [ ]:
# 같은 해시값이 2장 이상인 이미지만 남깁니다. keep=False: 중복된 이미지를 모두 남김
dup_df = hash_df[hash_df.duplicated("hash", keep=False)]

# 묶음(해시값)마다 train, test 이미지가 각각 몇 장인지 셉니다.
# True/False 열을 만들어 더하면 True의 개수가 됩니다.
group_df = (
    dup_df.assign(is_train=dup_df["split"] == "train", is_test=dup_df["split"] == "test")
    .groupby("hash")
    .agg(n_train=("is_train", "sum"), n_test=("is_test", "sum"))
)
# 중복이 0개면 표가 비어서 개수 열의 자료형(dtype)이 숫자로 정해지지 않을 수 있으므로, 정수형(int)으로 변환합니다.
group_df = group_df.astype({"n_train": int, "n_test": int})
group_df["n_images"] = group_df["n_train"] + group_df["n_test"]

# 묶음의 종류를 정합니다.
group_df["kind"] = "train-test 사이"
group_df.loc[group_df["n_test"] == 0, "kind"] = "train 안"
group_df.loc[group_df["n_train"] == 0, "kind"] = "test 안"

print(f"중복 묶음: {len(group_df)}개 (묶음에 속한 이미지 {len(dup_df)}장)")

# 종류별 묶음 수와 이미지 수. 해당하는 묶음이 없는 종류도 0으로 보여 줍니다.
KINDS = ["train-test 사이", "train 안", "test 안"]
summary_df = pd.DataFrame(
    {
        "묶음_수": [int((group_df["kind"] == kind).sum()) for kind in KINDS],
        "이미지_수": [int(group_df.loc[group_df["kind"] == kind, "n_images"].sum()) for kind in KINDS],
    },
    index=pd.Index(KINDS, name="kind"),
)
summary_df

## 3-3. 정리

- 3-1에서 train 232장, test 842장, 모두 1074장의 해시값을 계산했고, 1074개가 모두 달랐습니다.
- 3-2에서 해시값이 같은 묶음은 train 안, test 안, train-test 사이 모두 0개였습니다.
- 따라서 크기, 색 모드, 픽셀 값까지 완전히 같은 이미지는 train과 test 어디에도 없습니다.
- 살짝 변형해서 찍은 사진은 이번 검사 대상이 아니므로, 그런 사진이 있는지는 이 결과로 알 수 없습니다.

# 4. 이미지당 어노테이션 수와 라벨 누락

train 이미지 232장마다 어노테이션이 몇 개인지 세고, 라벨이 빠진 사진이 있는지 확인합니다. 3번에서 완전히 같은 이미지는 없었으므로, 232장을 모두 그대로 셉니다.

라벨 누락 여부는 아래 순서로 판단합니다.

1. 파일명에서 알약 코드를 뽑고, 어노테이션의 `category_id`가 그 코드 안에 있는지 봅니다. 2-3에서 파일명과 사진을 대조해 추정한 내용("`K-` 뒤의 6자리 숫자는 알약 종류를 나타내는 코드")을 코드로 확인하는 단계입니다.
2. 이미지마다 파일명 코드와 어노테이션을 비교해서, 차이가 나는 사진을 후보로 뽑습니다.
3. 후보 사진에 bbox를 그려서, 박스가 없는 알약이 있는지 눈으로 확인합니다.
4. 차이가 없던 사진도 몇 장 골라 같은 방식으로 확인합니다. 파일명 자체가 사진과 다른 경우는 2번 비교로 찾을 수 없기 때문입니다.

## 4-1. 이미지당 어노테이션 수

어노테이션이 1개, 2개, 3개, ... 인 이미지가 각각 몇 장인지 셉니다.

In [ ]:
# 이미지마다 어노테이션 수를 셉니다. (ann_df는 bbox 1개 = 1행)
ann_count = ann_df.groupby("file_name").size()

# 전체 train 이미지 목록 기준으로 맞춰서, 어노테이션이 없는 이미지가 있으면 0개로 셉니다.
ann_count = ann_count.reindex([p.name for p in train_img_paths], fill_value=0)

# 0개부터 최대 개수까지 모두 표시합니다. 해당하는 이미지가 없는 개수도 0장으로 보여서 빈 구간이 드러납니다.
dist = ann_count.value_counts().reindex(range(ann_count.max() + 1), fill_value=0)
print(dist.rename_axis("어노테이션 수").rename("이미지 수").to_string())
print(f"합계: 이미지 {dist.sum()}장, 어노테이션 {ann_count.sum()}개")

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(dist.index.astype(str), dist.values, width=0.5, color="#2a78d6")
ax.bar_label(bars, padding=3, color="#52514e")

ax.set_title("Images by number of annotations", loc="left")
ax.set_xlabel("annotations per image", color="#898781")
ax.set_ylabel("number of images", color="#898781")
ax.set_ylim(0, dist.max() * 1.15)  # 막대 위 숫자가 잘리지 않도록 여유를 둡니다.
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

## 4-2. 파일명에서 알약 코드 뽑기

train 파일명은 `K-001900-016548-019607-029451_0_2_0_2_70_000_200.png`처럼 생겼습니다. 첫 번째 `_` 앞부분을 `-`로 나누면 `K` 뒤에 숫자 조각이 남습니다. 이 조각을 알약 코드로 보고, 앞의 0을 뺀 숫자로 바꿉니다(`001900` → `1900`).

이 해석은 가설입니다. 아래 출력으로 판단합니다.

- 나눈 결과 표: 파일명이 의도대로 나뉘었는지 봅니다.
- 파일명의 코드 수: 코드가 몇 개인 파일이 각각 몇 장인지 셉니다.
- `category_id`가 파일명 코드 안에 있는지: 어노테이션마다 확인합니다. 모두 들어 있으면 가설을 뒷받침하는 근거가 되지만, 파일명에 적힌 약이 사진에 실제로 있는지까지 보장하지는 않습니다.

In [ ]:
def codes_from_name(file_name):
    """파일명 앞부분의 알약 코드를 숫자 목록으로 바꾼다.

    예: "K-001900-016548-019607-029451_0_2_..." -> [1900, 16548, 19607, 29451]
    형식이 예상과 다르면 빈 목록을 돌려준다.
    """
    head = file_name.split("_")[0]   # "K-001900-016548-019607-029451"
    parts = head.split("-")          # ["K", "001900", "016548", "019607", "029451"]
    if parts[0] != "K" or not all(p.isdigit() for p in parts[1:]):
        return []
    return [int(p) for p in parts[1:]]  # int("001900") -> 1900 (앞의 0이 빠짐)


# 이미지 1장 = 1행인 표를 만듭니다.
img_df = pd.DataFrame({"file_name": [p.name for p in train_img_paths]})
img_df["codes"] = img_df["file_name"].apply(codes_from_name)
img_df["n_codes"] = img_df["codes"].apply(len)

# 형식이 예상과 달라서 코드를 뽑지 못한 파일
bad_names = img_df[img_df["n_codes"] == 0]
print(f"코드를 뽑지 못한 파일명: {len(bad_names)}개")
if len(bad_names):
    print("  예시:", bad_names["file_name"].head().tolist())

print()
print(img_df["n_codes"].value_counts().sort_index().rename_axis("파일명의 코드 수").rename("이미지 수").to_string())

img_df.head()

In [ ]:
# 어노테이션마다 자기 이미지의 파일명 코드 안에 category_id가 있는지 봅니다.
codes_by_name = dict(zip(img_df["file_name"], img_df["codes"]))
in_codes = pd.Series(
    [cid in codes_by_name.get(name, []) for cid, name in zip(ann_df["category_id"], ann_df["file_name"])],
    index=ann_df.index,
)

print(f"category_id가 파일명 코드 안에 있음 : {in_codes.sum()}개")
print(f"category_id가 파일명 코드 밖에 있음 : {(~in_codes).sum()}개")

# 코드 밖에 있는 어노테이션이 있을 때만 앞의 10개를 표로 보여 줍니다.
not_in_codes_df = ann_df.loc[~in_codes, ["file_name", "category_id", "dl_idx", "dl_name", "json_path"]]
if len(not_in_codes_df) > 0:
    display(not_in_codes_df.head(10))

## 4-3. 파일명 코드와 어노테이션 비교

이미지마다 파일명의 코드와 어노테이션의 `category_id`를 비교합니다.

- `n_codes`: 파일명의 코드 수, `n_anns`: 어노테이션 수
- `missing_codes`: 파일명에는 있는데 어노테이션에는 없는 코드
- `compare`: 개수 비교 결과 (같음 / 어노테이션이 적음 / 어노테이션이 많음)

개수가 같아도 코드가 다를 수 있어서(예: 같은 알약에 라벨이 두 번, 다른 알약은 0번), 개수가 다르거나 `missing_codes`가 있는 사진을 모두 후보로 봅니다.

In [ ]:
# 이미지마다 실제 어노테이션의 category_id 목록을 모읍니다.
ann_ids_by_name = ann_df.groupby("file_name")["category_id"].apply(list)
img_df["ann_ids"] = [ann_ids_by_name.get(name, []) for name in img_df["file_name"]]
img_df["n_anns"] = img_df["ann_ids"].apply(len)

# 파일명에는 있는데 어노테이션에는 없는 코드
img_df["missing_codes"] = [sorted(set(c) - set(a)) for c, a in zip(img_df["codes"], img_df["ann_ids"])]

# 개수 비교
img_df["compare"] = "같음"
img_df.loc[img_df["n_anns"] < img_df["n_codes"], "compare"] = "어노테이션이 적음"
img_df.loc[img_df["n_anns"] > img_df["n_codes"], "compare"] = "어노테이션이 많음"

has_missing = img_df["missing_codes"].apply(len) > 0
print(img_df["compare"].value_counts().rename_axis("개수 비교").rename("이미지 수").to_string())
print(f"개수는 같지만 빠진 코드가 있는 이미지: {((img_df['compare'] == '같음') & has_missing).sum()}장")

# 후보: 개수가 다르거나, 빠진 코드가 있는 이미지
cand_df = img_df[(img_df["compare"] != "같음") | has_missing]
print(f"라벨 누락 확인 후보: {len(cand_df)}장")

cand_df[["file_name", "n_codes", "n_anns", "missing_codes", "compare"]]

## 4-4. 후보 사진에 bbox 그려 보기

4-3의 후보 사진에 어노테이션의 bbox와 `category_id`를 그립니다. 박스가 없는 알약이 보이면 라벨이 빠진 것입니다.

- 사진 위에는 파일명, 코드 수와 어노테이션 수(`codes / anns`), 빠진 코드(`missing`)를 적습니다.
- 한 번에 12장씩 보여 줍니다. 다음 12장을 보려면 `START` 값을 12, 24, ...로 바꿔 다시 실행합니다.

In [ ]:
def show_thumbs(paths, captions, boxes=None, title="", ncols=4, thumb=400):
    """이미지를 작게 줄여 격자로 보여 준다.

    paths   : 이미지 경로 목록
    captions: 각 이미지 위에 적을 글 목록
    boxes   : 각 이미지에 그릴 bbox 목록. 이미지마다 [(x, y, w, h, 글), ...] 형태이고, 좌표는 원본 픽셀 기준
    thumb   : 줄인 이미지의 긴 변 길이(픽셀). 그리는 시간과 메모리를 줄이기 위해 줄여서 그린다.
    """
    if len(paths) == 0:
        print("그릴 이미지가 없습니다.")
        return

    nrows = (len(paths) + ncols - 1) // ncols
    # squeeze=False: 이미지가 1장이어도 axes를 항상 2차원 배열로 받는다.
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 3.8 * nrows), layout="constrained", squeeze=False)
    for ax in axes.flat:
        ax.axis("off")

    for i, (ax, path) in enumerate(zip(axes.flat, paths)):
        with Image.open(path) as img:
            w, h = img.size                 # 원본 크기 (bbox 좌표의 기준)
            img.thumbnail((thumb, thumb))   # 비율을 유지한 채 긴 변이 thumb 픽셀이 되도록 줄임
            # extent: 줄인 이미지를 원본 크기의 좌표 위에 펼쳐 그린다. 그래서 bbox는 원본 좌표 그대로 그리면 된다.
            ax.imshow(img, extent=(0, w, h, 0))
        ax.set_title(captions[i], fontsize=7)

        if boxes is not None:
            for x, y, bw, bh, label in boxes[i]:
                ax.add_patch(plt.Rectangle((x, y), bw, bh, fill=False, edgecolor="#e5484d", linewidth=1.5))
                # 글자는 박스 안쪽 왼쪽 위에 적어서 사진 밖으로 나가지 않게 한다.
                ax.text(x + 8, y + 8, label, fontsize=7, color="white", va="top",
                        bbox=dict(facecolor="#e5484d", edgecolor="none", pad=1))

    fig.suptitle(title, x=0.01, ha="left")
    plt.show()

def boxes_for(file_name):
    """이미지 하나의 bbox 목록을 show_thumbs에 넘길 형태로 만든다. 형식이 유효한 bbox만 그린다."""
    rows = ann_df[(ann_df["file_name"] == file_name) & ann_df["bbox_ok"]]
    return [(r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h, str(r.category_id)) for r in rows.itertuples()]

def show_with_boxes(rows, title, ncols=4):
    """img_df의 행들을 받아 bbox를 그린 사진을 보여 준다."""
    show_thumbs(
        paths=[path_by_name[name] for name in rows["file_name"]],
        captions=[
            r.file_name.replace("_", "\n_", 1) + f"\ncodes {r.n_codes} / anns {r.n_anns}, missing {r.missing_codes}"
            for r in rows.itertuples()
        ],
        boxes=[boxes_for(name) for name in rows["file_name"]],
        title=title,
        ncols=ncols,
    )

path_by_name = {p.name: p for p in train_img_paths}
START = 0   # 몇 번째 후보부터 볼지
PAGE = 12   # 한 번에 볼 사진 수

page_df = cand_df.iloc[START:START + PAGE]
show_with_boxes(page_df, f"Label check candidates: {START + 1}-{START + len(page_df)} of {len(cand_df)}")

## 4-5. 후보가 아닌 사진도 확인

4-3에서 파일명 코드와 어노테이션이 일치한 사진 중 6장을 무작위로 골라 같은 방식으로 그립니다(seed 고정).

파일명 자체가 사진과 다르면(예: 파일명에는 코드가 3개인데 사진에는 알약이 4개) 4-3 비교로는 찾을 수 없습니다. 여기서도 박스 없는 알약이 보이면, 파일명 비교만으로는 부족하다는 뜻이므로 전체 이미지를 확인하는 방법으로 바꿔야 합니다.

In [ ]:
ok_df = img_df.drop(cand_df.index)  # 후보가 아닌 이미지
picked = random.Random(42).sample(list(ok_df.index), min(6, len(ok_df)))
show_with_boxes(ok_df.loc[picked], "Non-candidates: 6 random samples (seed=42)", ncols=3)

## 4-6. 정리

- 4-1: 어노테이션이 2개인 이미지 7장, 3개 151장, 4개 74장입니다(합계 232장, 어노테이션 763개). 0개나 1개인 이미지는 없습니다.
- 4-2: 232장 모두 파일명에서 코드를 뽑을 수 있었고, 코드가 3개인 이미지 157장, 4개인 이미지 75장입니다. 어노테이션 763개 모두 `category_id`가 자기 이미지의 파일명 코드 안에 있었습니다. 2-3의 추정("`K-` 뒤의 6자리 숫자는 알약 종류를 나타내는 코드")과 맞는 결과입니다.
- 4-3: 224장은 파일명 코드와 어노테이션이 일치했고, 8장은 어노테이션이 1개씩 적었습니다. 개수는 같은데 코드가 다른 이미지와 어노테이션이 더 많은 이미지는 없었습니다.
    - 파일명 코드 수의 합(3 × 157 + 4 × 75 = 771개)에서 어노테이션 수(763개)를 빼면 8개로, 빠진 코드 수와 같습니다.
    - 빠진 코드는 3351이 4장, 20014, 32310, 33880, 4543이 각각 1장입니다.
- 4-4: 후보 8장 모두에서 박스가 없는 알약이 보였습니다. 즉 train 이미지 8장에서 알약 1개씩, 모두 8개의 라벨이 빠져 있습니다.
- 4-5: 후보가 아닌 이미지 중 무작위 6장은 보이는 알약 모두에 박스가 있었습니다. 다만 6장만 본 것이라, 나머지 이미지 전체에 라벨 누락이 없다고 보장하지는 않습니다.
- 빠진 라벨 8개를 학습 전에 어떻게 처리할지(라벨 추가, 해당 이미지 제외 등)는 아직 정하지 않았습니다.

# 5. bbox 값 검사

4번에서는 있어야 할 bbox가 있는지(라벨 누락)를 확인했습니다. 이번에는 있는 bbox의 값이 올바른지 확인합니다. 1-6에서는 bbox가 값 4개짜리인지(형식)만 확인했고, 값 자체는 아직 보지 않았습니다.

### 이 작업을 하는 이유

- 대회 평가 지표가 mAP@[0.75:0.95]입니다. 가장 낮은 기준도 IoU 0.75라서, 예측 박스가 정답 박스와 매우 정확히 겹쳐야 점수가 납니다. 정답 bbox 값이 잘못되어 있으면 모델이 잘못된 위치를 학습하고, 검증 점수도 실제와 다르게 나옵니다.
- 대회 설명에 "제공된 데이터에는 크고 작은 오류들이 포함되어 있다"고 되어 있습니다.
- bbox 크기 분포는 학습할 때 입력 이미지 크기를 정하는 데 참고할 수 있습니다.

### 용어 설명

- IoU(Intersection over Union): 두 박스가 겹친 부분의 넓이를 두 박스를 합친 부분의 넓이로 나눈 값입니다. 0이면 전혀 겹치지 않고, 1이면 완전히 같은 박스입니다.
- AP(Average Precision): 한 클래스의 예측을 신뢰도 순으로 늘어놓았을 때 그려지는 정밀도-재현율 곡선 아래의 넓이입니다.
- mAP@[0.75:0.95]: IoU 기준을 0.75에서 0.95까지 여러 값으로 바꿔 가며 클래스별 AP를 구하고, 이를 모두 평균한 값입니다.

### 진행 방법

먼저 bbox 좌표의 기준이 되는 이미지 크기를 확인하고, 각 단계의 결과를 보고 다음 단계를 정합니다. 확인할 후보는 다음과 같습니다.

- bbox가 이미지 범위 안에 있는지, 너비와 높이가 0보다 큰지
- `area`가 `너비 × 높이`와 같은지
- bbox 크기와 가로세로 비율이 다른 bbox와 동떨어지지 않는지(이상치)
- 같은 이미지 안에서 bbox끼리 크게 겹치지 않는지

## 5-1. 이미지 크기 확인

bbox는 원본 이미지의 픽셀 좌표로 적혀 있습니다. 그래서 bbox가 이미지 범위 안에 있는지 보려면 기준이 되는 이미지 크기가 정확해야 합니다. JSON에 적힌 이미지 크기(`img_width`, `img_height`)와 실제 이미지 파일의 크기가 같은지 확인합니다.

- JSON마다 이미지 크기가 따로 적혀 있으므로, 어노테이션 763개마다 비교합니다.
- 실제 이미지 크기가 몇 종류인지도 함께 셉니다. 2-3에서는 샘플 12장만 976x1280이었습니다.

In [ ]:
# 실제 이미지 파일의 크기를 읽습니다.
# Image.open은 파일 앞부분(헤더)만 읽고 픽셀은 읽지 않으므로, 크기만 얻을 때는 빠릅니다.
real_size = {}
for p in train_img_paths:
    with Image.open(p) as img:
        real_size[p.name] = img.size  # (너비, 높이)

print("실제 이미지 크기 (train)")
for (w, h), n in Counter(real_size.values()).most_common():
    print(f"  {w}x{h}: {n}장")
print()

# 어노테이션마다 JSON에 적힌 이미지 크기와 실제 파일 크기를 비교합니다.
size_df = ann_df[["file_name", "img_width", "img_height"]].copy()
size_df["real_width"] = [real_size.get(name, (None, None))[0] for name in size_df["file_name"]]
size_df["real_height"] = [real_size.get(name, (None, None))[1] for name in size_df["file_name"]]
size_df["size_match"] = (size_df["img_width"] == size_df["real_width"]) & (size_df["img_height"] == size_df["real_height"])

print(f"JSON 크기와 실제 크기가 같은 어노테이션 : {size_df['size_match'].sum()}개")
print(f"JSON 크기와 실제 크기가 다른 어노테이션 : {(~size_df['size_match']).sum()}개")

# 다른 어노테이션이 있을 때만 앞의 10개를 표로 보여 줍니다.
# if 블록 안에서는 마지막 줄의 표가 자동으로 출력되지 않으므로 display()로 출력합니다.
mismatch_df = size_df[~size_df["size_match"]]
if len(mismatch_df) > 0:
    display(mismatch_df.head(10))

## 5-2. bbox 범위와 크기 확인

5-1에서 train 이미지 232장이 모두 976x1280이고, JSON에 적힌 이미지 크기도 763개 모두 실제 파일과 같았습니다. 그래서 JSON의 `img_width`, `img_height`를 기준으로 bbox를 검사합니다.

bbox는 `[x, y, w, h]` 형식입니다. (x, y)는 박스 왼쪽 위 꼭짓점의 좌표, w와 h는 너비와 높이이며, 이미지의 왼쪽 위가 (0, 0)입니다. 따라서 오른쪽 아래 꼭짓점은 (x + w, y + h)입니다. 4-4, 4-5 그림은 이 해석으로 박스를 그렸고, 박스가 알약 위치에 그려졌습니다.

확인하는 항목은 다음과 같습니다.

- 숫자가 아닌 값: 숫자로 바꿀 수 없는 값(예: 문자)이 들어 있는 bbox
- 크기 0 이하: 너비나 높이가 0 이하인 bbox
- 이미지 밖: 왼쪽 위 꼭짓점이 0보다 작거나, 오른쪽 아래 꼭짓점이 이미지 너비·높이보다 큰 bbox

좌표와 크기의 최솟값, 최댓값도 함께 출력해서 bbox가 이미지 안 어느 범위에 있는지 봅니다.

In [ ]:
bbox_cols = ["bbox_x", "bbox_y", "bbox_w", "bbox_h"]

# 형식이 유효한 bbox(값 4개짜리 리스트)만 검사합니다. 1-6에서 763개 모두 유효했습니다.
box_df = ann_df.loc[ann_df["bbox_ok"], ["file_name", "category_id", "img_width", "img_height"] + bbox_cols].copy()

# 숫자가 아닌 값이 있는지 확인합니다. 숫자로 바꿀 수 없는 값은 NaN(빈 값)이 됩니다.
numeric = box_df[bbox_cols].apply(pd.to_numeric, errors="coerce")
not_number = numeric.isna().any(axis=1)
box_df[bbox_cols] = numeric

# 오른쪽 아래 꼭짓점 좌표
box_df["x2"] = box_df["bbox_x"] + box_df["bbox_w"]
box_df["y2"] = box_df["bbox_y"] + box_df["bbox_h"]

# 너비나 높이가 0 이하인 bbox
non_positive = (box_df["bbox_w"] <= 0) | (box_df["bbox_h"] <= 0)

# 이미지 밖으로 나간 bbox: 왼쪽 위가 0보다 작거나, 오른쪽 아래가 이미지 너비·높이보다 큼
outside = (
    (box_df["bbox_x"] < 0) | (box_df["bbox_y"] < 0)
    | (box_df["x2"] > box_df["img_width"]) | (box_df["y2"] > box_df["img_height"])
)

print(f"검사한 bbox    : {len(box_df)}개")
print(f"숫자가 아닌 값 : {not_number.sum()}개")
print(f"크기 0 이하    : {non_positive.sum()}개")
print(f"이미지 밖      : {outside.sum()}개")
print()

# 좌표와 크기의 최솟값, 최댓값. 이미지 크기(976x1280)와 비교해 범위를 볼 수 있습니다.
print(box_df[["bbox_x", "bbox_y", "x2", "y2", "bbox_w", "bbox_h"]].agg(["min", "max"]).to_string())

# 하나라도 걸린 bbox가 있으면 앞의 10개를 보여 줍니다. (없으면 빈 표)
problem = not_number | non_positive | outside
box_df[problem].head(10)

## 5-3. 이미지 밖 bbox가 있는 사진 보기

5-2에서 이미지 밖으로 나간 bbox가 1개 있었습니다. 이 bbox가 가리키려던 알약이 사진의 어디에 있는지 확인하려고 사진을 그립니다.

- 이미지 밖 bbox는 사진 위에 그릴 수 없으므로, 그 값을 사진 제목에 `outside: category_id [x, y, w, h]` 형태로 적고, 이미지 안에 있는 bbox만 그립니다.
- 파일명 앞부분(첫 번째 `_` 앞)이 같은 이미지도 함께 그립니다. 2-3의 추정대로라면 같은 알약들이 찍힌 사진이므로, 다른 사진에서 같은 `category_id`의 박스가 어느 알약에 그려지는지 비교할 수 있습니다.

In [ ]:
# 이미지 밖으로 나간 bbox가 있는 이미지와, 파일명 앞부분(첫 번째 _ 앞)이 같은 이미지를 모두 모읍니다.
out_df = box_df[outside]
prefixes = {name.split("_")[0] for name in out_df["file_name"]}
same_prefix_names = sorted(p.name for p in train_img_paths if p.name.split("_")[0] in prefixes)


def inside_boxes(file_name):
    """이미지 하나에서 이미지 안에 있는 bbox만 show_thumbs에 넘길 형태로 만든다."""
    rows = box_df[(box_df["file_name"] == file_name) & ~outside]
    return [(r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h, str(r.category_id)) for r in rows.itertuples()]


def outside_text(file_name):
    """이미지 밖 bbox의 category_id와 값을 사진 제목에 붙일 글로 만든다. 없으면 빈 글."""
    rows = out_df[out_df["file_name"] == file_name]
    return "".join(
        f"\noutside: {r.category_id} [{r.bbox_x}, {r.bbox_y}, {r.bbox_w}, {r.bbox_h}]" for r in rows.itertuples()
    )


# 이미지 밖 bbox는 사진 위에 그릴 수 없으므로, 값만 사진 제목에 적고 이미지 안 bbox만 그립니다.
show_thumbs(
    paths=[path_by_name[name] for name in same_prefix_names],
    captions=[name.replace("_", "\n_", 1) + outside_text(name) for name in same_prefix_names],
    boxes=[inside_boxes(name) for name in same_prefix_names],
    title="Images with an out-of-image bbox, and images with the same file name prefix",
    ncols=3,
)

## 5-4. x 값 후보 그려 보기

5-3 그림에서 `_75_` 사진의 노란 타원형 알약에는 박스가 없고, `_90_` 사진에서는 같은 모양의 알약에 18357 박스가 있습니다. 이 bbox는 y와 너비·높이가 다른 bbox들의 범위 안에 있고, x만 이미지 너비(976)를 넘습니다.

x 값 6567은 네 자리인데 이미지 너비는 세 자리입니다. 그래서 "입력할 때 숫자가 한 자리 더 들어갔다"고 가정하고, 6567에서 한 자리씩 뺀 값(567, 667, 657, 656)을 후보로 만들어 박스를 각각 그려 봅니다.

- 이 가정은 확인되지 않은 것입니다. 그림은 후보가 알약 위치와 맞는지 보기 위한 것이고, 데이터는 고치지 않습니다.
- 이 bbox를 어떻게 처리할지(값 수정, 학습에서 제외 등)는 팀과 상의해 정합니다.

In [ ]:
# 이미지 밖 bbox의 x 값에서 숫자를 한 자리씩 뺀 값을 후보로 만듭니다.
# 예: 6567 -> 567(첫째 자리를 뺌), 667(둘째), 657(셋째), 656(넷째)
for r in out_df.itertuples():
    x_str = str(int(r.bbox_x))
    cand_xs = sorted({int(x_str[:i] + x_str[i + 1:]) for i in range(len(x_str))})

    # 같은 사진을 후보 수만큼 그리고, 사진마다 이미지 안 bbox와 후보 박스 하나를 함께 그립니다.
    show_thumbs(
        paths=[path_by_name[r.file_name]] * len(cand_xs),
        captions=[
            f"candidate x = {cx}\n[{cx}, {r.bbox_y}, {r.bbox_w}, {r.bbox_h}], x2 = {cx + r.bbox_w}"
            for cx in cand_xs
        ],
        boxes=[
            inside_boxes(r.file_name) + [(cx, r.bbox_y, r.bbox_w, r.bbox_h, f"{r.category_id} (x={cx})")]
            for cx in cand_xs
        ],
        title=f"category_id {r.category_id}: candidate x (original {x_str})",
        ncols=len(cand_xs),
    )

### 오류 기록: 이미지 밖 bbox 1개

- 대상: `K-003351-016262-018357_0_2_0_2_75_000_200.png`의 `category_id` 18357 bbox `[6567, 625, 311, 315]`
- 문제: x 값이 이미지 너비(976)보다 커서 박스가 이미지 밖에 있습니다(x2 = 6878). y, 너비, 높이는 다른 bbox들의 범위 안에 있습니다.
- 사진 확인(5-3): `_75_` 사진의 노란 타원형 알약에는 박스가 없고, `_90_` 사진에서는 같은 모양의 알약에 18357 박스가 있습니다.
- 후보 확인(5-4): 6567에서 숫자를 한 자리씩 뺀 후보 중, x = 567일 때 박스가 그 알약 위치와 맞아 보였습니다(눈으로 판단).
- 처리 방법(x를 567로 수정, 학습에서 제외 등)은 팀원과 상의해 정합니다. 노트북의 데이터는 고치지 않았습니다.

## 5-5. 클래스별 bbox 영역 모아 보기

### 이 작업을 하는 이유

- 4-3에서는 파일명의 코드와 라벨의 `category_id`가 서로 맞는지만 비교했습니다. 그래서 파일명과 라벨이 함께 틀린 경우는 찾을 수 없습니다. 예를 들어 파일명과 라벨은 A인데 사진 속 알약은 B인 경우입니다.
- 같은 `category_id`로 라벨된 bbox 영역을 잘라 한곳에 모으면, 대부분 같은 모양의 알약이어야 합니다. 모양이 다른 알약이 섞여 있으면, 라벨의 클래스가 잘못됐을 가능성이 있는 후보입니다.
- 잘라 낸 영역을 보면 bbox가 알약을 제대로 감싸는지(알약이 잘리거나, 박스가 알약보다 너무 큰지)도 함께 볼 수 있습니다.
- 사진 전체를 한 장씩 보는 것보다 같은 클래스끼리 비교하기가 빠릅니다.

### 진행 방법

먼저 클래스 수와 클래스별 bbox 수를 세어서, 한 번에 몇 개 클래스를 어떤 크기로 보여 줄지 정합니다. 5-2에서 찾은 이미지 밖 bbox 1개는 잘라 낼 수 없으므로 뺍니다.

In [ ]:
# 이미지 안에 있는 bbox만 사용합니다. (5-2의 이미지 밖 bbox 1개 제외)
crop_df = box_df[~outside]

# 클래스(category_id)별 bbox 수
crop_count = crop_df["category_id"].value_counts()

print(f"잘라 낼 bbox : {len(crop_df)}개")
print(f"클래스 수    : {crop_count.size}개")
print(f"클래스별 bbox 수: 최소 {crop_count.min()}개, 최대 {crop_count.max()}개, 중앙값 {crop_count.median():.1f}개")
print()

# bbox 수가 같은 클래스가 몇 개인지 셉니다. (예: bbox가 3개인 클래스가 몇 개인지)
print(crop_count.value_counts().sort_index().rename_axis("클래스별 bbox 수").rename("클래스 수").to_string())

### 그림 저장

클래스마다 bbox 영역을 잘라 작게 줄인 뒤, 한 줄에 12개씩 모아 그립니다.

- 그림 한 묶음이 한 클래스이고, 묶음 제목에 `category_id`와 bbox 수(`n`)를 적습니다.
- 칸마다 왼쪽 위의 빨간 번호는 그 클래스 안에서의 순서입니다. 이상한 영역을 찾으면 `category_id`와 번호로 원래 사진을 찾을 수 있습니다.
- 56개 클래스를 한 그림에 넣으면 노트북 용량이 크게 늘어나서, 8개 클래스씩 7개 그림으로 나눠 `notebooks/eda/class_bbox_crops/` 폴더에 저장합니다.
- 노트북에는 저장한 파일 이름과, 그 파일에 들어 있는 클래스의 약 이름만 출력합니다.
- 같은 이름의 그림 파일이 이미 있으면 다시 만들지 않고 건너뜁니다. 코드나 데이터를 바꿔서 그림을 다시 만들어야 할 때는, 폴더의 그림 파일을 지운 뒤 실행합니다.

In [ ]:
CELL = 100   # 잘라 낸 영역 하나를 넣을 칸의 크기(픽셀)
NCOLS = 12   # 한 줄에 넣을 칸 수

# 모든 bbox 영역을 잘라 칸 크기에 맞게 줄여 둡니다. 이미지는 한 장씩 한 번만 엽니다.
crops = {}  # crop_df의 행 번호 -> 줄인 영역 이미지
for file_name, rows in crop_df.groupby("file_name"):
    with Image.open(path_by_name[file_name]) as img:
        img = img.convert("RGB")
        for r in rows.itertuples():
            box = tuple(int(v) for v in (r.bbox_x, r.bbox_y, r.bbox_x + r.bbox_w, r.bbox_y + r.bbox_h))
            crop = img.crop(box)                  # (왼쪽, 위, 오른쪽, 아래) 영역을 잘라 냄
            crop.thumbnail((CELL - 4, CELL - 4))  # 비율을 유지한 채 칸 안에 들어가게 줄임
            crops[r.Index] = crop

print(f"잘라 낸 영역: {len(crops)}개")


def class_montage(cat_id):
    """한 클래스의 잘라 낸 영역을 한 줄에 NCOLS개씩 붙인 이미지를 만든다. 칸마다 순서 번호를 적는다."""
    idx = crop_df.index[crop_df["category_id"] == cat_id]
    nrows = (len(idx) + NCOLS - 1) // NCOLS
    canvas = Image.new("RGB", (NCOLS * CELL, nrows * CELL), "white")
    draw = ImageDraw.Draw(canvas)
    for k, i in enumerate(idx):
        c = crops[i]
        col, row = k % NCOLS, k // NCOLS
        # 칸 가운데에 붙입니다.
        canvas.paste(c, (col * CELL + (CELL - c.width) // 2, row * CELL + (CELL - c.height) // 2))
        draw.text((col * CELL + 3, row * CELL + 2), str(k), fill="red")
    return canvas

In [ ]:
CROP_DIR = Path("eda/class_bbox_crops")   # 저장 폴더 (노트북 위치인 notebooks/ 기준)

# 폴더가 없으면 저장할 때 오류가 나므로, 여기서 먼저 확인하고 멈춥니다.
assert CROP_DIR.exists(), f"저장 폴더를 찾을 수 없습니다: {CROP_DIR} (notebooks/eda/ 아래에 폴더를 만들어 주세요)"

class_ids = sorted(crop_count.index)                         # category_id 순서
class_names = ann_df.groupby("category_id")["dl_name"].first()
CLASS_PAGE = 8                                                # 그림 하나에 넣을 클래스 수

for page_no, start in enumerate(range(0, len(class_ids), CLASS_PAGE), 1):
    page_ids = class_ids[start:start + CLASS_PAGE]
    out_path = CROP_DIR / f"class_bbox_crops_{page_no:02d}.png"

    if out_path.exists():
        # 같은 이름의 그림 파일이 이미 있으면 다시 만들지 않습니다.
        status = "이미 있어서 건너뜀"
    else:
        montages = [class_montage(cid) for cid in page_ids]
        heights = [m.height for m in montages]

        # 클래스마다 줄 수가 달라서, 각 묶음의 높이를 줄 수에 맞춰 나눕니다.
        fig, axes = plt.subplots(
            len(page_ids), 1,
            figsize=(12, sum(heights) / 100 + 0.4 * len(page_ids)),
            gridspec_kw={"height_ratios": heights},
            layout="constrained",
            squeeze=False,
        )
        for ax, cid, m in zip(axes[:, 0], page_ids, montages):
            ax.imshow(m)
            ax.set_title(f"category_id {cid} (n={crop_count[cid]})", loc="left", fontsize=9)
            ax.axis("off")

        fig.savefig(out_path, dpi=100)
        plt.close(fig)  # 파일로만 저장하고, 노트북에는 그림을 표시하지 않습니다.
        status = "새로 저장함"

    print(f"{out_path.as_posix()}  (클래스 {start + 1}~{start + len(page_ids)}, {status})")
    for cid in page_ids:
        print(f"    {cid}: {class_names[cid]} ({crop_count[cid]}개)")

### 5-5 그림에서 다르게 보이는 칸

`notebooks/eda/class_bbox_crops/`의 그림 7장을 눈으로 보고, 같은 클래스의 다른 칸과 다르게 생긴 칸을 골랐습니다. 원인(라벨이 잘못 붙었는지, bbox가 다른 곳에 그려졌는지)은 다음 단계에서 원래 사진으로 확인합니다.

| category_id | 번호 | 같은 클래스의 다른 칸 | 이 칸 |
|---|---|---|---|
| 3351 | 97 | 주황색 원형 "HT" | 알약 없이 배경만 보임 |
| 3351 | 129 | 주황색 원형 "HT" | 흰색 원형 알약 |
| 16548 | 3 | 노란 원형 "GB" | 갈색 타원형 "S300" 알약 (33009 칸들과 비슷) |
| 20238 | 7 | 분홍빛 원형 "75" | 주황색 원형 "HT" 알약 (3351 칸들과 비슷) |
| 27733 | 10 | 흰색 타원형 "A1" 로고 | 흰색 긴 알약 "337" (35206 칸들과 비슷) |
| 35206 | 29 | 흰색 긴 알약 "337" | 흰색 타원형 "A1" 로고 (27733 칸들과 비슷) |

### 글자가 두 종류지만 표에 넣지 않은 클래스

아래 두 클래스는 칸마다 보이는 글자가 다르지만, 한 알약의 앞뒷면으로 보여서(추정) 위 표에 넣지 않았습니다.

| category_id | 칸에 보이는 글자 | 앞뒷면으로 본 근거 |
|---|---|---|
| 3483 | "G40", "SK"(9~11번, 15~17번) | 56개 클래스 중 "SK" 글자가 있는 알약이 다른 클래스에 보이지 않습니다. 두 글자의 칸 모두 색(녹색)과 모양(긴 타원형)이 같습니다. |
| 30308 | 로고, 글자 각인(3~5번) | 56개 클래스 중 같은 글자 각인이 있는 알약이 다른 클래스에 보이지 않습니다. 두 종류의 칸 모두 색(주황색)과 모양(긴 타원형)이 같습니다. |

JSON의 `print_front`, `print_back`(식별문자 앞, 뒤) 값과 비교하면 앞뒷면인지 확인할 수 있습니다.

- 5-2에서 찾은 이미지 밖 bbox 1개(18357)는 잘라 낼 수 없어서 그림에 포함하지 않았습니다. 그래서 그림의 18357은 13개 중 12개입니다.

## 5-6. 다르게 보인 칸의 원래 사진 보기

5-5 그림에서 다르게 보인 6칸이 어느 사진의 어느 bbox인지 찾고, 그 사진 전체에 bbox를 모두 그려서 문제의 bbox가 실제로 어디에 그려져 있는지 확인합니다.

- 표: 칸마다 이미지 파일명, bbox 값, JSON 파일 경로를 보여 줍니다. JSON 경로는 나중에 팀에서 이 라벨을 고칠 때 원본 파일을 찾는 데 씁니다.
- 그림: 사진마다 이미지 안에 있는 bbox를 모두 그립니다. 다르게 보인 칸의 bbox는 글자에 `check #번호`를 붙여 구분합니다.
- 사진 제목에는 파일명에서 뽑은 코드(`codes`)를 적습니다. 2-3의 추정대로라면 그 사진에 있어야 할 알약 종류이므로, 박스의 `category_id`와 비교할 수 있습니다.

In [ ]:
# 5-5 그림에서 다르게 보인 칸: (category_id, 그 클래스 안의 번호)
suspect_cells = [(3351, 97), (3351, 129), (16548, 3), (20238, 7), (27733, 10), (35206, 29)]

suspect_rows = []
for cid, k in suspect_cells:
    # class_montage와 같은 방법으로 순서를 매기므로, k번째 행이 그림의 k번 칸입니다.
    row_id = crop_df.index[crop_df["category_id"] == cid][k]
    r = crop_df.loc[row_id]
    suspect_rows.append({
        "row_id": row_id,
        "category_id": cid,
        "번호": k,
        "file_name": r["file_name"],
        "bbox": [r["bbox_x"], r["bbox_y"], r["bbox_w"], r["bbox_h"]],
        "json_path": ann_df.loc[row_id, "json_path"],
    })
suspect_df = pd.DataFrame(suspect_rows).set_index("row_id")
display(suspect_df)


def boxes_with_check(file_name):
    """이미지 하나의 bbox 목록을 만든다. 다르게 보인 칸의 bbox에는 'check #번호'를 붙인다."""
    boxes = []
    for r in crop_df[crop_df["file_name"] == file_name].itertuples():
        label = str(r.category_id)
        if r.Index in suspect_df.index:
            label += f" (check #{suspect_df.loc[r.Index, '번호']})"
        boxes.append((r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h, label))
    return boxes


# 같은 사진에 다르게 보인 칸이 2개 이상 있을 수 있으므로, 순서를 유지하며 파일명 중복을 없앱니다.
suspect_names = list(dict.fromkeys(suspect_df["file_name"]))

show_thumbs(
    paths=[path_by_name[name] for name in suspect_names],
    captions=[name.replace("_", "\n_", 1) + f"\ncodes {codes_from_name(name)}" for name in suspect_names],
    boxes=[boxes_with_check(name) for name in suspect_names],
    title="Images containing the cells that looked different in 5-5",
    ncols=3,
)

### 5-6 그림에서 보이는 것

다르게 보인 6칸은 사진 5장에 들어 있었습니다(27733 #10과 35206 #29가 같은 사진). 아래는 그림을 눈으로 보고 읽은 내용입니다. 박스 글자가 겹쳐 보이는 부분은 아래 좌표 표로 확인합니다.

### bbox 좌표 표

| 칸 | 사진 | 보이는 것 |
|---|---|---|
| 3351 #97 | `K-003351-020014-020238_0_2_0_2_90_000_200.png` | 3351 박스가 주황색 "HT" 알약 바로 왼쪽의 빈 곳에 있고, "HT" 알약에는 박스가 없습니다. |
| 3351 #129 | `K-003351-029667-031863_0_2_0_2_70_000_200.png` | 흰색 알약 위에 31863 박스와 3351 박스의 글자가 겹쳐 보이고, 아래쪽 "HT" 알약에는 박스가 없습니다. |
| 16548 #3 | `K-001900-016548-019607-033009_0_2_0_2_70_000_200.png` | "S300" 알약 위에 33009 박스와 16548 박스의 글자가 겹쳐 보이고, 노란 "GB" 알약에는 박스가 없습니다. |
| 20238 #7 | `K-003351-020238-031863_0_2_0_2_70_000_200.png` | 20238 박스가 "HT" 알약 위에 있고, 분홍빛 "75" 알약에는 박스가 없습니다. 4-3에서 어노테이션이 3개였는데 박스는 2개만 보여서, 3351 박스가 20238 박스와 겹쳐 있는 것으로 보입니다. |
| 27733 #10, 35206 #29 | `K-003483-025367-027733-035206_0_2_0_2_75_000_200.png` | 27733 박스가 "337" 알약에, 35206 박스가 "A1" 알약에 있습니다. 박스 위치는 각 알약에 맞고, 두 라벨이 서로 바뀐 것으로 보입니다. |

유형으로 나누면 다음과 같습니다(추정).

1. 박스 위치가 어긋남: 3351 #97
2. 박스가 다른 알약의 박스와 같은 곳에 겹쳐 있음: 3351 #129, 16548 #3, 20238 #7. 원래 알약에는 박스가 없습니다.
3. 두 알약의 라벨이 서로 바뀜: 27733 #10, 35206 #29

- 유형 1, 2의 사진은 4-3에서 파일명 코드와 라벨 코드가 일치한("같음") 사진입니다. 코드는 맞지만 실제로는 박스가 없는 알약이 있어서, 4-3 비교로는 찾을 수 없었던 경우입니다.

그림에서 박스 글자가 겹쳐 보이는 곳이 있어서, 위 사진 5장에 있는 bbox를 모두 표로 봅니다. 같은 사진 안에서 좌표가 같거나 거의 같은 bbox가 있으면, 박스 두 개가 같은 알약 위에 겹쳐 있다는 뜻입니다. `check` 열은 5-5에서 다르게 보인 칸의 번호입니다.

In [ ]:
detail_df = crop_df.loc[
    crop_df["file_name"].isin(suspect_names),
    ["file_name", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h"],
].copy()
detail_df["check"] = [f"#{suspect_df.loc[i, '번호']}" if i in suspect_df.index else "" for i in detail_df.index]

# 사진별로 모으고, 사진 안에서는 x 좌표 순서로 정렬합니다.
detail_df.sort_values(["file_name", "bbox_x"])

### 좌표 확인 결과

- 유형 2의 세 칸은 같은 사진 안의 다른 bbox와 좌표가 완전히 같았습니다. 박스 두 개가 같은 알약 위에 겹쳐 있습니다.

| 칸 | 좌표가 같은 bbox | 좌표 `[x, y, w, h]` |
|---|---|---|
| 3351 #129 | 31863 | `[178, 249, 205, 203]` |
| 16548 #3 | 33009 | `[88, 255, 366, 209]` |
| 20238 #7 | 3351 | `[446, 844, 190, 191]` |

- 겹친 박스 아래 알약은 각각 흰색 원형, "S300", "HT" 알약이고, 5-5의 클래스별 모양으로 보면 31863, 33009, 3351에 해당합니다. 따라서 3351 #129, 16548 #3, 20238 #7이 잘못된 위치의 bbox로 보이며, 이 bbox들이 원래 가리켜야 할 알약("HT", "GB", "75")에는 박스가 없습니다.
- 3351 #97은 같은 사진의 다른 bbox와 겹치지 않았습니다(유형 1).
- 27733 #10과 35206 #29는 좌표가 서로 다르고, 각자 다른 알약 위에 있습니다. 라벨이 바뀌었는지는 좌표로는 알 수 없어서, 5-6 그림의 판단(유형 3)을 따릅니다.

## 5-7. 같은 사진 안 bbox 겹침 검사

### 이 작업을 하는 이유

- 5-6에서 다른 알약의 bbox와 좌표가 완전히 같은 bbox 3개를 찾았습니다(유형 2). 이 bbox들이 원래 가리켜야 할 알약에는 박스가 없었습니다.
- 5-6은 5-5 그림을 눈으로 봐서 찾은 것이라, 겹친 두 알약이 비슷하게 생긴 경우는 놓쳤을 수 있습니다. 4-3도 라벨의 코드만 비교하므로 이런 경우를 찾지 못합니다.
- 좌표로 비교하면 알약 생김새와 관계없이 찾을 수 있습니다.

### 방법

- 같은 사진 안의 bbox를 두 개씩 짝지어 IoU를 계산합니다. IoU는 5번 섹션 소개의 용어 설명을 참고합니다.
- IoU = 1이면 좌표가 완전히 같은 bbox입니다. 2-3 샘플에서는 알약이 서로 떨어져 놓여 있었으므로, 서로 다른 알약의 bbox라면 IoU가 0이거나 아주 작아야 합니다.
- `known` 열은 5-6에서 이미 찾은 bbox가 들어 있는 쌍인지 표시합니다.

In [ ]:
def iou(a, b):
    """두 bbox [x, y, w, h]의 IoU(겹친 넓이 / 두 박스를 합친 넓이)를 계산한다."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    # 겹친 영역의 너비와 높이. 겹치지 않으면 0입니다.
    inter_w = max(0, min(ax + aw, bx + bw) - max(ax, bx))
    inter_h = max(0, min(ay + ah, by + bh) - max(ay, by))
    inter = inter_w * inter_h
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


# 사진마다 bbox를 두 개씩 짝지어 IoU를 계산합니다. (5-2의 box_df: bbox 763개 전체)
pair_rows = []
for file_name, rows in box_df.groupby("file_name"):
    boxes = list(rows.itertuples())
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            a, b = boxes[i], boxes[j]
            bbox_a = [a.bbox_x, a.bbox_y, a.bbox_w, a.bbox_h]
            bbox_b = [b.bbox_x, b.bbox_y, b.bbox_w, b.bbox_h]
            pair_rows.append({
                "file_name": file_name,
                "row_a": a.Index, "row_b": b.Index,
                "id_a": a.category_id, "id_b": b.category_id,
                "iou": round(iou(bbox_a, bbox_b), 3),
                "bbox_a": bbox_a, "bbox_b": bbox_b,
            })
pair_df = pd.DataFrame(pair_rows)

# 5-6에서 이미 찾은 bbox가 들어 있는 쌍인지 표시합니다.
pair_df["known"] = pair_df["row_a"].isin(suspect_df.index) | pair_df["row_b"].isin(suspect_df.index)

same = pair_df["iou"] == 1
print(f"검사한 bbox 쌍                  : {len(pair_df)}개 (사진 {pair_df['file_name'].nunique()}장)")
print(f"좌표가 완전히 같은 쌍 (IoU = 1)   : {same.sum()}개")
print(f"조금이라도 겹치는 쌍 (0 < IoU < 1): {((pair_df['iou'] > 0) & ~same).sum()}개")
print(f"겹치지 않는 쌍 (IoU = 0)          : {(pair_df['iou'] == 0).sum()}개")

# 겹치는 쌍이 있을 때만, IoU가 큰 순서로 앞의 20개를 보여 줍니다.
overlap_df = pair_df[pair_df["iou"] > 0].sort_values("iou", ascending=False)
if len(overlap_df) > 0:
    display(overlap_df[["file_name", "id_a", "id_b", "iou", "bbox_a", "bbox_b", "known"]].head(20))

### 조금 겹치는 쌍 사진으로 보기

0 < IoU < 1인 쌍이 있는 사진을 그립니다. 왼쪽은 사진 전체, 오른쪽은 두 bbox를 모두 덮는 영역에 여백을 더해 확대한 그림입니다. 알약끼리 실제로 맞닿아 있는지, 박스 모서리만 겹치는지 확인합니다.

In [ ]:
# 0 < IoU < 1인 쌍만 고릅니다.
partial_df = pair_df[(pair_df["iou"] > 0) & (pair_df["iou"] < 1)]
MARGIN = 60  # 확대할 때 두 bbox 바깥으로 더 보여 줄 여백(픽셀)

for r in partial_df.itertuples():
    with Image.open(path_by_name[r.file_name]) as img:
        img = img.convert("RGB")  # 파일을 닫은 뒤에도 쓸 수 있도록 픽셀을 읽어 둡니다.

    fig, axes = plt.subplots(1, 2, figsize=(10, 6), layout="constrained")
    for ax in axes:
        ax.imshow(img)
        # 5-3의 inside_boxes: 이 사진에서 이미지 안에 있는 bbox를 모두 가져옵니다.
        for x, y, w, h, label in inside_boxes(r.file_name):
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor="#e5484d", linewidth=1.5))
            # clip_on=True: 확대 그림에서 범위 밖 글자가 그림 바깥에 찍히지 않게 합니다.
            ax.text(x + 8, y + 8, label, fontsize=8, color="white", va="top", clip_on=True,
                    bbox=dict(facecolor="#e5484d", edgecolor="none", pad=1))
        ax.axis("off")

    # 오른쪽 그림: 두 bbox를 모두 덮는 영역에 여백을 더해 확대합니다. 이미지 밖으로는 넘지 않게 합니다.
    (xa, ya, wa, ha), (xb, yb, wb, hb) = r.bbox_a, r.bbox_b
    left = max(0, min(xa, xb) - MARGIN)
    right = min(img.width, max(xa + wa, xb + wb) + MARGIN)
    top = max(0, min(ya, yb) - MARGIN)
    bottom = min(img.height, max(ya + ha, yb + hb) + MARGIN)
    axes[1].set_xlim(left, right)
    axes[1].set_ylim(bottom, top)  # 이미지 좌표는 아래로 갈수록 y가 커져서, 아래쪽 값을 먼저 넣습니다.

    axes[0].set_title("full image", fontsize=9)
    axes[1].set_title(f"zoom: {r.id_a} & {r.id_b}, IoU {r.iou}", fontsize=9)
    fig.suptitle(r.file_name, x=0.01, ha="left", fontsize=10)
    plt.show()

### 5-7 결과

- bbox 쌍 904개(사진 232장)를 모두 비교했습니다. 4-1의 어노테이션 수로 계산한 쌍의 수(7×1 + 151×3 + 74×6 = 904개)와 같습니다.
- 좌표가 완전히 같은 쌍(IoU = 1)은 3개이고, 모두 5-6에서 찾은 쌍입니다(3351 #129와 31863, 16548 #3과 33009, 20238 #7과 3351). 새로 찾은 쌍은 없습니다.
- 조금 겹치는 쌍은 2개이고, IoU가 0.021, 0.016으로 매우 작습니다. 두 쌍 모두 33880("JIC")과 35206("337")이며, 같은 조합(`K-003351-033880-035206`)의 `_75_`, `_90_` 사진입니다.
    - 확대 그림을 보면, 두 긴 알약이 맞닿을 만큼 가까이 나란히 놓여 있고, 각 박스는 자기 알약을 감싸고 있습니다. 박스가 알약보다 조금 크게 그려져 있어서, 두 박스가 맞닿는 가장자리가 조금 겹친 것입니다. 라벨 오류는 아닌 것으로 봅니다.
- 나머지 899쌍은 겹치지 않습니다.

## 5-8. JSON 결측치 확인

### 이 작업을 하는 이유

- 지금까지는 JSON에서 꺼낸 값이 있다고 보고 분석했습니다. 1-6의 `ann_df`도 값이 비어 있는지는 세어 보지 않았습니다.
- 학습과 제출에 쓰는 값(bbox, `category_id`)이나 지금까지 분석에 쓴 값에 빈 곳이 있으면, 앞의 결과를 다시 봐야 할 수 있습니다.
- 다음 단계에서 `area`를 bbox의 너비 × 높이와 비교하려면, `area` 값이 모두 있어야 합니다.

### 방법

- `annotations`, `categories`의 모든 항목과, `images`의 항목 중 지금까지 쓴 5개(`file_name`, `width`, `height`, `dl_idx`, `dl_name`)를 확인합니다. `images`의 나머지 항목은 이후 JSON `images` 항목 전체를 볼 때 확인합니다.
- 먼저 JSON마다 `images`, `annotations`, `categories` 안에 항목 묶음이 몇 개씩 있는지 셉니다. 1-5에서는 `images`와 `annotations`만 셌습니다.
- 항목마다 키 자체가 없는 경우(키 없음)와, 키는 있지만 값이 `None`, 빈 문자열, 빈 리스트처럼 비어 있는 경우(빈 값)를 따로 셉니다. 숫자 0은 값이 있는 것으로 봅니다.
- 1-1 명세의 필수 여부(M: 필수, O: 선택, C: 조건부)를 함께 보여 줍니다. 선택 항목은 비어 있어도 명세상 문제가 아닙니다. 1-4에서 `segmentation`은 빈 리스트였습니다.

In [ ]:
# 확인할 항목과 대회 명세(1-1)의 필수 여부 (M: 필수, O: 선택, C: 조건부)
CHECK_FIELDS = {
    "images": {"file_name": "M", "width": "M", "height": "M", "dl_idx": "M", "dl_name": "M"},
    "annotations": {"area": "M", "iscrowd": "O", "bbox": "M", "category_id": "M",
                    "ignore": "O", "segmentation": "O", "image_id": "M", "id": "M"},
    "categories": {"supercategory": "M", "id": "M", "name": "C"},
}


def is_empty(value):
    """None, 빈 문자열(공백만 있는 문자열 포함), 빈 리스트, 빈 딕셔너리를 빈 값으로 본다."""
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    if isinstance(value, (list, dict)):
        return len(value) == 0
    return False


# JSON마다 images, annotations, categories 안에 항목 묶음이 몇 개씩 있는지 셉니다.
for part in CHECK_FIELDS:
    sizes = Counter(len(data.get(part) or []) for _, data in loaded)
    print(f"{part}: " + ", ".join(f"묶음 {n}개인 JSON {c}개" for n, c in sorted(sizes.items())))
print()

# 항목마다 키 없음과 빈 값을 셉니다.
missing_rows = []
for part, fields in CHECK_FIELDS.items():
    for field, required in fields.items():
        n_checked = n_no_key = n_empty = 0
        for _, data in loaded:
            for item in data.get(part) or []:
                n_checked += 1
                if field not in item:
                    n_no_key += 1
                elif is_empty(item[field]):
                    n_empty += 1
        missing_rows.append({
            "부분": part, "항목": field, "명세 필수 여부": required,
            "확인한 수": n_checked, "키 없음": n_no_key, "빈 값": n_empty,
        })
missing_df = pd.DataFrame(missing_rows)
missing_df

### 5-8 결과

- JSON 763개 모두 `images`, `annotations`, `categories`에 항목 묶음이 1개씩 있습니다.
- 확인한 16개 항목 모두 키가 빠진 JSON은 없습니다.
- 빈 값은 `segmentation`에만 있고, 763개 모두 빈 리스트입니다. 명세상 선택 항목(O)이라 문제가 아닙니다.
- 따라서 확인한 범위(`annotations`, `categories` 전체와 `images`의 5개 항목)에는 결측치가 없습니다. `area`도 763개 모두 값이 있어서, 다음 단계에서 너비 × 높이와 비교할 수 있습니다.

## 5-9. `area` 확인

### 이 작업을 하는 이유

- JSON의 `area`는 대회 명세에서 "바운딩박스 면적"입니다. 따라서 bbox의 너비 × 높이와 같아야 합니다.
- 1-4에서 본 JSON 하나는 `area` 35910이 bbox의 너비 × 높이(189 × 190)와 같았습니다. 하지만 763개 전체는 확인하지 않았습니다.
- `area` 자체는 학습에 쓰지 않는 값입니다. 다만 두 값이 다르면 bbox와 `area` 중 하나가 잘못 기록됐다는 뜻이라, 5-2의 범위 검사로는 찾을 수 없는 bbox 값 오류의 단서가 될 수 있습니다.

### 방법

- `area`는 1-6의 `ann_df`에 넣지 않았던 값이라, 1-3에서 읽은 `loaded`에서 다시 꺼냅니다. `ann_df`를 만들 때와 같은 순서로 꺼내서 행 순서를 맞춥니다.
- 5-2의 `box_df`(숫자로 바꾼 bbox 763개)로 너비 × 높이를 계산해 `area`와 비교합니다.

In [ ]:
# JSON의 area를 1-6의 ann_df와 같은 순서로 꺼냅니다.
# to_rows도 loaded를 앞에서부터, JSON 안의 annotations를 순서대로 돌았기 때문에 행 순서가 같습니다.
area_list = [ann.get("area") for _, data in loaded for ann in data.get("annotations", [])]
assert len(area_list) == len(ann_df), "area 개수와 ann_df 행 수가 다릅니다."
json_area = pd.Series(area_list, index=ann_df.index)

area_df = box_df[["file_name", "category_id", "bbox_w", "bbox_h"]].copy()
area_df["area"] = pd.to_numeric(json_area.loc[area_df.index], errors="coerce")
area_df["w_x_h"] = area_df["bbox_w"] * area_df["bbox_h"]   # bbox의 너비 × 높이
area_df["diff"] = area_df["area"] - area_df["w_x_h"]       # area - 너비 × 높이

same_area = area_df["diff"] == 0
print(f"비교한 bbox                     : {len(area_df)}개")
print(f"area = 너비 × 높이               : {same_area.sum()}개")
print(f"area ≠ 너비 × 높이               : {(~same_area).sum()}개")

# 다른 bbox가 있을 때만, 차이가 큰 순서로 앞의 10개를 보여 줍니다.
diff_df = area_df[~same_area]
if len(diff_df) > 0:
    display(diff_df.reindex(diff_df["diff"].abs().sort_values(ascending=False).index).head(10))

## 5-10. `area`가 다른 bbox 확인

5-9에서 `area`와 너비 × 높이가 다른 bbox 6개를 두 가지로 나눠 확인합니다.

### "숫자 한 자리가 잘못 적혔다"고 가정한 이유

5-9 표의 차이(`diff`)가 23200, −3000, 2660, 1890, −300, 60처럼 딱 떨어지는 숫자라서, 숫자 한 자리가 잘못 적힌 경우인지 계산해 봤습니다. 그 결과 6개가 아래 A, B로 나뉘었습니다. 5-4에서도 x 값 6567에 숫자가 한 자리 더 들어간 것으로 보이는 경우가 있었습니다. 이 데이터에 숫자 한 자리 단위의 입력 오류가 있을 수 있다는 근거로 봅니다.

### A. 너비나 높이의 숫자 한 자리를 바꾸면 `area`와 맞는 경우 (3개): 사진으로 확인

- 행 363: 높이 241 → 341로 보면 232 × 341 = 79112 = `area`
- 행 370: 너비 322 → 332로 보면 332 × 266 = 88312 = `area`
- 행 452: 높이 182 → 192로 보면 189 × 192 = 36288 = `area`

계산만으로는 bbox와 `area` 중 어느 쪽이 틀렸는지 알 수 없어서, 아래 코드로 사진에 그려 판단합니다.

- 왼쪽 그림은 JSON에 적힌 현재 bbox, 오른쪽 그림은 `area`에 맞춘 bbox입니다.
- 왼쪽 위 꼭짓점(x, y)은 그대로 두고 너비나 높이만 바꾸므로, 두 박스는 오른쪽 또는 아래쪽 끝만 다릅니다. 그래서 박스 주변을 확대해서 그립니다.
- 박스가 알약을 제대로 감싸는 쪽을 맞는 값으로 봅니다. 데이터는 고치지 않습니다.

### B. `area`의 숫자 한 자리만 너비 × 높이와 다른 경우 (3개): 그리지 않음

| 행 | 사진 | category_id | 너비 × 높이 | `area` | 다른 자리 |
|---|---|---|---|---|---|
| 541 | `K-003351-033880-038162_0_2_0_2_70_000_200.png` | 38162 | 251 × 259 = 65009 | 62009 | 천의 자리 (5 → 2) |
| 540 | `K-003351-033880-038162_0_2_0_2_70_000_200.png` | 33880 | 309 × 420 = 129780 | 129480 | 백의 자리 (7 → 4) |
| 80 | `K-002483-012081-012778-025438_0_2_0_2_90_000_200.png` | 12778 | 317 × 508 = 161036 | 161096 | 십의 자리 (3 → 9) |

- 너비나 높이의 숫자를 한 자리 바꿔서는 `area`가 나오지 않습니다. `area`는 넓이 숫자 하나라서 그것만으로는 박스 모양을 정할 수 없고, 그래서 그릴 후보 박스가 없습니다.
- 그래서 `area`가 잘못 적힌 것으로 봅니다(추정). 5-5 그림에서도 38162, 33880, 12778 클래스에는 다르게 보인 칸이 없었습니다.
- `area`는 학습에 쓰지 않는 값이라 학습에는 영향이 없습니다.
- 541과 540은 같은 사진에 있는 bbox입니다.

In [ ]:
def one_digit_alternatives(value, other, area):
    """value의 숫자 한 자리를 다른 숫자로 바꿨을 때 value × other == area가 되는 값을 모두 찾는다."""
    s = str(int(value))
    found = []
    for i in range(len(s)):
        for d in "0123456789":
            if d == s[i]:
                continue
            cand = int(s[:i] + d + s[i + 1:])
            if cand > 0 and cand * other == area:
                found.append(cand)
    return found


MARGIN = 80  # 확대할 때 박스 바깥으로 더 보여 줄 여백(픽셀)

# A: 너비나 높이의 숫자 한 자리를 바꾸면 area와 맞는 bbox만 고릅니다. 후보가 없는 B는 그리지 않습니다.
targets = []
for idx, r in diff_df.iterrows():
    b = box_df.loc[idx]  # 5-2의 box_df에서 같은 행의 x, y를 가져옵니다.
    x, y, w, h = int(b["bbox_x"]), int(b["bbox_y"]), int(b["bbox_w"]), int(b["bbox_h"])
    area = int(r["area"])
    alts = [(f"area-based: w {w} -> {nw}", (x, y, nw, h)) for nw in one_digit_alternatives(w, h, area)]
    alts += [(f"area-based: h {h} -> {nh}", (x, y, w, nh)) for nh in one_digit_alternatives(h, w, area)]
    if alts:
        targets.append((idx, r["file_name"], r["category_id"], area, [("current bbox", (x, y, w, h))] + alts))

print(f"area가 다른 bbox {len(diff_df)}개 중, 사진으로 확인할 bbox(A): {len(targets)}개")

for idx, file_name, cat_id, area, panels in targets:
    with Image.open(path_by_name[file_name]) as img:
        img = img.convert("RGB")  # 파일을 닫은 뒤에도 쓸 수 있도록 픽셀을 읽어 둡니다.

    # 확대 범위: 그릴 박스를 모두 덮는 영역에 여백을 더하고, 이미지 밖으로는 넘지 않게 합니다.
    left = max(0, min(p[1][0] for p in panels) - MARGIN)
    top = max(0, min(p[1][1] for p in panels) - MARGIN)
    right = min(img.width, max(p[1][0] + p[1][2] for p in panels) + MARGIN)
    bottom = min(img.height, max(p[1][1] + p[1][3] for p in panels) + MARGIN)

    fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels), 4.5), layout="constrained", squeeze=False)
    for ax, (label, (bx, by, bw, bh)) in zip(axes[0], panels):
        ax.imshow(img)
        ax.add_patch(plt.Rectangle((bx, by), bw, bh, fill=False, edgecolor="#e5484d", linewidth=2))
        ax.set_xlim(left, right)
        ax.set_ylim(bottom, top)  # 이미지 좌표는 아래로 갈수록 y가 커져서, 아래쪽 값을 먼저 넣습니다.
        ax.set_title(f"{label}\n[{bx}, {by}, {bw}, {bh}], w x h = {bw * bh}", fontsize=9)
        ax.axis("off")

    fig.suptitle(f"row {idx}: {file_name} / category_id {cat_id} / area {area}", x=0.01, ha="left", fontsize=10)
    plt.show()

### 5-10 결과

A의 3개를 사진으로 본 결과입니다(눈으로 판단).

| 행 | 사진 | category_id | 판단 |
|---|---|---|---|
| 363 | `K-003351-019232-029667_0_2_1_2_70_000_200.png` | 19232 | 현재 bbox(높이 241)는 알약 아래 부분이 박스 밖으로 나오고, `area`에 맞춘 bbox(높이 341)는 알약 전체를 감쌉니다. bbox의 높이가 잘못 적힌 것으로 봅니다. |
| 370 | `K-003351-019232-035206_0_2_0_2_90_000_200.png` | 19232 | 현재 bbox(너비 322)와 `area`에 맞춘 bbox(너비 332) 모두 알약 전체를 감싸서, 어느 쪽이 맞는지 구분하기 어렵습니다. |
| 452 | `K-003351-021325-033880_0_2_0_2_70_000_200.png` | 3351 | 현재 bbox(높이 182)는 아래 선이 알약 아래 옆면에 거의 붙어 있고, `area`에 맞춘 bbox(높이 192)는 약간의 여백을 두고 감쌉니다. 차이가 10픽셀이라 구분하기 어렵습니다. |

- 363은 두 박스의 IoU가 약 0.71(241 ÷ 341)로, 평가 기준 0.75보다 낮아서, 두 박스 중 어느 쪽을 정답으로 쓰느냐에 따라 같은 예측도 맞음과 틀림이 갈립니다.
- 370과 452는 두 박스의 IoU가 각각 약 0.97, 0.95로, 어느 쪽이 맞든 차이가 작습니다.
- 처리 방법(bbox 수정 등)은 팀원과 상의해 정합니다. 노트북의 데이터는 고치지 않았습니다.

## 5-11. 정리

5번 섹션의 제목은 "bbox 값 검사"이지만, 진행하면서 bbox 영역 안의 알약이 라벨 클래스와 맞는지(5-5, 5-6)와 JSON 결측치(5-8)도 함께 확인했습니다.

### 확인한 것

| 단계 | 검사 | 결과 |
|---|---|---|
| 5-1 | 이미지 크기 | train 232장 모두 976x1280이고, JSON에 적힌 크기도 763개 모두 실제 파일과 같습니다. |
| 5-2 ~ 5-4 | bbox 범위 | 숫자가 아닌 값과 크기 0 이하인 bbox는 없습니다. 이미지 밖 bbox가 1개 있습니다(18357, x = 6567). x = 567일 때 알약 위치와 맞아 보였습니다. |
| 5-5, 5-6 | bbox 영역의 알약과 라벨 | 다르게 보인 6칸을 확인했습니다. 박스 위치가 어긋난 bbox 1개(3351 #97), 다른 알약의 박스와 좌표가 완전히 같은 bbox 3개(3351 #129, 16548 #3, 20238 #7), 라벨이 서로 바뀐 것으로 보이는 bbox 2개(27733 #10, 35206 #29)입니다. |
| 5-7 | 같은 사진 안 bbox 겹침 | 좌표가 완전히 같은 쌍은 5-6에서 찾은 3쌍뿐입니다. 조금 겹치는 2쌍은 가까이 놓인 알약의 박스 가장자리가 겹친 것입니다. |
| 5-8 | JSON 결측치 | 확인한 16개 항목에 결측치가 없습니다. `segmentation`은 모두 빈 리스트이지만 선택 항목입니다. |
| 5-9, 5-10 | `area`와 너비 × 높이 | 6개가 달랐습니다. bbox 높이가 잘못 적힌 것으로 보이는 1개(행 363), 어느 쪽이 맞는지 구분하기 어려운 2개(행 370, 452), `area`가 잘못 적힌 것으로 보이는 3개(행 541, 540, 80)입니다. |

### 4번과 5번 결과 합산: 라벨 문제가 있는 train 사진

| 문제 | 알약(bbox) 수 | 사진 수 | 확인한 곳 |
|---|---|---|---|
| 알약에 박스가 없음 (라벨 누락) | 8개 | 8장 | 4-3, 4-4 |
| 알약에 박스가 없음 (박스가 빈 곳이나 다른 알약 위에 있음) | 4개 | 4장 | 5-6 |
| 알약에 박스가 보이지 않음 (이미지 밖 bbox) | 1개 | 1장 | 5-2 ~ 5-4 |
| 박스 높이가 알약보다 짧음 | 1개 | 1장 | 5-9, 5-10 |
| 두 알약의 라벨이 서로 바뀜 | 2개 | 1장 | 5-5, 5-6 |
| `area`가 너비 × 높이와 다름 (어느 쪽이 맞는지 구분하기 어려움, 행 370, 452) | 2개 | 2장 | 5-9, 5-10 |
| `area`가 잘못 적힘 (행 541, 540, 80) | 3개 | 2장 | 5-9, 5-10 |

- 위 문제가 하나라도 있는 사진은 train 232장 중 19장이고, 서로 겹치는 사진은 없습니다.
- 그중 올바른 박스가 없는 알약은 13개입니다(라벨 누락 8개, 5-6의 4개, 이미지 밖 1개).
- 5-6, 5-10의 일부 판단은 그림을 눈으로 보고 한 것입니다.

### 하지 않은 것

- bbox 크기와 가로세로 비율의 이상치 확인, bbox 크기 분포 확인(5번 소개의 확인 후보)
- JSON `images` 항목 중 5개를 뺀 나머지 항목의 결측치 확인
- 4-5에서 무작위로 본 6장을 뺀, 후보가 아닌 사진 전체에 박스를 그려 보는 확인

### 정해야 할 것

- 위 라벨 문제의 처리 방법(값 수정, 학습에서 제외 등)은 팀원과 상의해 정합니다. 노트북의 데이터는 고치지 않았습니다.

# 6. 클래스 분포

train 라벨의 클래스(알약 종류)가 몇 개이고, 클래스마다 bbox와 이미지가 얼마나 있는지 봅니다.

### 이 작업을 하는 이유

- 팀 규칙에 따라 클래스 목록(`classes.py`)을 만들어야 합니다. 클래스 목록은 `category_id`와 약 이름을 짝지은 것이라, 먼저 이 짝이 데이터 안에서 일관되게 적혀 있는지 확인해야 합니다.
- 5-5에서 클래스별 bbox 수가 최소 3개, 최대 153개로 차이가 컸습니다. 이렇게 차이가 크면 모델이 bbox가 많은 클래스 위주로 학습될 수 있습니다. 불균형 정도를 알아야 대응 방법(샘플링, 증강 등)을 정할 수 있습니다.
- 대회 설명에 test에는 train에 없는 클래스가 있다고 되어 있습니다. train에 있는 클래스를 정확히 알아 두어야 합니다.

### 세는 기준

- 노트북의 데이터는 고치지 않았으므로, 어노테이션에 적힌 그대로(763개) 셉니다.
- 4번과 5번에서 찾은 라벨 오류 중 클래스별 개수를 바꾸는 것은 라벨 누락뿐입니다. 위치가 틀리거나 두 라벨이 바뀐 경우는 라벨 개수 자체는 그대로입니다. 그래서 누락된 8개(3351이 4개, 20014·32310·33880·4543이 각 1개)만큼 실제보다 적게 셉니다.

## 6-1. `category_id`와 약 이름의 짝 확인

1-6에서 미뤄 둔 비교입니다. JSON 안에는 클래스를 나타내는 값이 세 군데 적혀 있습니다.

- `annotations`의 `category_id`
- `images`의 `dl_idx`(알약 식별자)와 `dl_name`(제품명)
- `categories`의 `id`와 `name`

대회 설명에 따르면 `category_id`는 `dl_idx` 값을 정수로 바꿔 만들었습니다. 세 곳의 값이 서로 같은지, 그리고 `category_id` 하나에 약 이름이 하나씩만 대응하는지 763개 전체에서 확인합니다.

In [ ]:
# annotations, images, categories에서 클래스와 관련된 값을 ann_df와 같은 순서로 모읍니다.
cat_rows = []
for _, data in loaded:
    img = (data.get("images") or [{}])[0]
    cat = (data.get("categories") or [{}])[0]
    for ann in data.get("annotations", []):
        cat_rows.append({
            "category_id": ann.get("category_id"),
            "dl_idx": img.get("dl_idx"),
            "dl_name": img.get("dl_name"),
            "cat_id": cat.get("id"),
            "cat_name": cat.get("name"),
        })
assert len(cat_rows) == len(ann_df), "모은 행 수와 ann_df 행 수가 다릅니다."
cat_df = pd.DataFrame(cat_rows, index=ann_df.index)

# 세 곳의 값이 같은지 비교합니다. dl_idx는 문자열이라 숫자로 바꿔 비교합니다.
id_match = pd.to_numeric(cat_df["dl_idx"], errors="coerce") == cat_df["category_id"]
catid_match = cat_df["cat_id"] == cat_df["category_id"]
name_match = cat_df["cat_name"] == cat_df["dl_name"]

print(f"category_id = dl_idx          : 같음 {id_match.sum()}개 / 다름 {(~id_match).sum()}개")
print(f"category_id = categories의 id : 같음 {catid_match.sum()}개 / 다름 {(~catid_match).sum()}개")
print(f"dl_name = categories의 name   : 같음 {name_match.sum()}개 / 다름 {(~name_match).sum()}개")
print()

# category_id 하나에 약 이름이 하나씩만 대응하는지, 반대로 약 이름 하나에 category_id가 하나씩만 대응하는지 셉니다.
names_per_id = cat_df.groupby("category_id")["dl_name"].nunique()
ids_per_name = cat_df.groupby("dl_name")["category_id"].nunique()
print(f"category_id 수: {names_per_id.size}개 / 약 이름이 2개 이상인 category_id: {(names_per_id > 1).sum()}개")
print(f"약 이름 수    : {ids_per_name.size}개 / category_id가 2개 이상인 약 이름: {(ids_per_name > 1).sum()}개")

# 하나라도 다른 행이 있을 때만 앞의 10개를 보여 줍니다.
mismatch_cat_df = cat_df[~(id_match & catid_match & name_match)]
if len(mismatch_cat_df) > 0:
    display(mismatch_cat_df.head(10))

### 6-1 결과

- 763개 모두 `category_id`, `dl_idx`, `categories`의 `id`가 같고, `dl_name`과 `categories`의 `name`도 같습니다.
- `category_id`는 56개, 약 이름도 56개이며, 서로 하나씩만 대응합니다. 따라서 클래스 목록은 `category_id`와 약 이름을 1:1로 짝지어 만들 수 있습니다.

## 6-2. 클래스별 bbox 수와 이미지 수

6-1에서 `category_id` 56개와 약 이름 56개가 하나씩 짝지어졌으므로, `category_id`를 클래스로 보고 셉니다.

- bbox 수: 그 클래스로 라벨된 bbox 개수입니다. 5-5와 달리, 이미지 밖 bbox 1개(18357)도 포함해 763개 전체를 셉니다.
- 이미지 수: 그 클래스가 들어 있는 train 이미지 수입니다. 한 사진에 같은 클래스가 두 번 라벨되어 있으면 bbox 수가 이미지 수보다 커집니다.
- 그래프의 세로축은 약 이름 대신 `category_id`입니다. matplotlib 기본 글꼴에 한글이 없기 때문입니다. 약 이름은 그래프 아래 표에서 봅니다.

In [ ]:
# 클래스별 약 이름, bbox 수, 이미지 수를 셉니다. bbox 수가 많은 순서로 정렬합니다.
class_df = (
    ann_df.groupby("category_id")
    .agg(dl_name=("dl_name", "first"), n_bbox=("file_name", "size"), n_image=("file_name", "nunique"))
    .sort_values("n_bbox", ascending=False)
)

total_bbox = class_df["n_bbox"].sum()
top_id = class_df.index[0]
print(f"클래스 수: {len(class_df)}개 / bbox 수: {total_bbox}개")
print(f"클래스별 bbox 수: 최소 {class_df['n_bbox'].min()}개, "
      f"중앙값 {class_df['n_bbox'].median():.1f}개, 최대 {class_df['n_bbox'].max()}개")
print(f"bbox가 가장 많은 클래스: {top_id} {class_df.loc[top_id, 'dl_name']} "
      f"({class_df.loc[top_id, 'n_bbox']}개, 전체의 {class_df.loc[top_id, 'n_bbox'] / total_bbox:.1%})")
print(f"bbox가 3개 이하인 클래스: {(class_df['n_bbox'] <= 3).sum()}개")
print(f"bbox 수와 이미지 수가 다른 클래스: {(class_df['n_bbox'] != class_df['n_image']).sum()}개")

# 가로 막대그래프: 아래에서 위로 갈수록 bbox가 많아지도록 정렬합니다.
order = class_df.sort_values("n_bbox")
fig, ax = plt.subplots(figsize=(8, 12))
ax.barh(order.index.astype(str), order["n_bbox"], height=0.6, color="#2a78d6")

# 값은 가장 긴 막대에만 적고, 나머지는 아래 표에서 봅니다.
ax.text(order["n_bbox"].iloc[-1] + 2, len(order) - 1, str(order["n_bbox"].iloc[-1]),
        va="center", fontsize=8, color="#52514e")

ax.set_title(f"Number of bboxes per class (train, {len(class_df)} classes)", loc="left")
ax.set_xlabel("number of bboxes", color="#898781")
ax.set_ylabel("category_id", color="#898781")
ax.tick_params(axis="y", labelsize=7)
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)  # 격자선을 막대 뒤로 보냅니다.
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

# 클래스별 표 (bbox 수가 많은 순서)
class_df

### 6-2 결과

- 클래스 56개, bbox 763개입니다. 클래스별 bbox 수는 최소 3개, 중앙값 9개, 최대 153개입니다.
- 3351(일양하이트린정 2mg)이 153개로 전체의 20.1%입니다. 그다음은 3483(45개), 35206(37개)이고, 나머지 53개 클래스는 23개 이하입니다.
- bbox가 3개인 클래스가 17개로, 전체 클래스의 약 30%입니다.
- 모든 클래스에서 bbox 수와 이미지 수가 같아서, 한 사진에 같은 클래스가 두 번 라벨된 경우는 없습니다. 따라서 3351은 train 이미지 232장 중 153장(약 66%)에 들어 있습니다.
- 4번의 라벨 누락을 반영하면, 3351은 4개, 20014·32310·33880·4543은 1개씩 실제보다 적게 세어졌습니다.

## 6-3. 클래스별 조합 수

학습/검증 분할 방법을 정하는 근거를 얻으려고, 클래스마다 몇 개의 조합에 나오는지 셉니다.

### 이 작업을 하는 이유

- 학습/검증 분할: 모델을 학습할 때는 train 이미지 일부를 떼어 검증 데이터로 남겨 두고, 학습에 쓰지 않은 사진으로 성능을 잽니다. test에는 라벨이 없어서, 제출 전에는 검증 데이터로 실험 결과를 비교합니다.
- 6-2에서 train 전체의 bbox가 3개뿐인 클래스(알약 종류)가 17개였습니다. 6-2에서 모든 클래스의 bbox 수와 이미지 수가 같았으므로, 이 17개 알약은 각각 train 사진 3장에만 나옵니다. 학습/검증 분할은 사진 단위로 하므로, 그 3장이 어디로 가느냐에 따라 문제가 생길 수 있습니다.
    - 3장 모두 학습으로 가면, 검증 사진에 그 알약이 없어서 잘 맞히는지 잴 수 없습니다.
    - 일부가 검증으로 가면, 학습 사진에는 그 알약이 1~2장만 남습니다.
    - 3장 모두 검증으로 가면, 모델은 학습 중에 그 알약을 한 번도 보지 못합니다.
- 1-2 트리에서 같은 조합 폴더 안에 파일명 끝(`_70_`, `_75_`, `_90_`)만 다른 JSON들이 있었습니다. 이 JSON들에 해당하는 사진이 학습과 검증 양쪽에 들어가면, 모델이 검증 사진과 거의 같은 사진을 이미 본 셈이 되어 검증 점수가 실제보다 높게 나올 수 있습니다(가설).

### 방법

- 2-3의 추정에 따라, 파일명 앞부분(첫 번째 `_` 앞, 예: `K-001900-016548-019607-029451`)이 같은 이미지를 같은 조합으로 봅니다.
- 먼저 조합이 몇 개이고, 조합마다 이미지가 몇 장인지 셉니다. 조합 수는 1-2 트리의 상위 폴더 수(114개)와 비교합니다.
- 다음으로 클래스마다 몇 개의 조합에 나오는지 세고, 조합 1개에만 나오는 클래스를 표로 보여 줍니다.

In [ ]:
# 파일명 앞부분(첫 번째 _ 앞)을 조합으로 봅니다.
img_combo = pd.Series([p.name.split("_")[0] for p in train_img_paths])

# 1) 조합 수와, 조합 하나에 이미지가 몇 장씩 있는지
combo_size = img_combo.value_counts()
print(f"조합 수: {combo_size.size}개 (train 이미지 {len(img_combo)}장)")
print(combo_size.value_counts().sort_index()
      .rename_axis("조합 하나의 이미지 수").rename("조합 수").to_string())
print()

# 2) 클래스마다 몇 개의 조합에 나오는지
ann_combo = ann_df["file_name"].str.split("_").str[0]
class_combo = ann_df.assign(combo=ann_combo).groupby("category_id")["combo"].nunique()
print(class_combo.value_counts().sort_index()
      .rename_axis("클래스가 나오는 조합 수").rename("클래스 수").to_string())
print()

# 6-2의 class_df에 조합 수를 붙인 새 표 (class_df는 그대로 둡니다)
combo_class_df = class_df.assign(n_combo=class_combo)
three_df = combo_class_df[combo_class_df["n_bbox"] == 3]
print(f"bbox가 3개인 클래스 {len(three_df)}개 중, 조합 1개에만 나오는 클래스: {(three_df['n_combo'] == 1).sum()}개")

# 조합 1개에만 나오는 클래스가 있을 때만, bbox 수가 많은 순서로 표를 보여 줍니다.
single_df = combo_class_df[combo_class_df["n_combo"] == 1]
if len(single_df) > 0:
    display(single_df[["dl_name", "n_bbox", "n_image", "n_combo"]])

### 6-3 결과

- 조합은 114개로, 1-2 트리의 상위 폴더 수(114개)와 같습니다. 조합 하나의 이미지 수는 1장 36개, 2장 38개, 3장 40개입니다.
- 조합 1개에만 나오는 클래스는 17개이고, 6-2에서 bbox가 3개였던 클래스 17개와 정확히 같습니다. 즉 이 17개 클래스는 모두 한 조합의 사진 3장에만 나옵니다.
- 3351은 114개 조합 중 89개에 나옵니다.
- 학습/검증 분할에서 뜻하는 것
    - 사진 단위로 무작위로 나누면, 이 17개 클래스는 같은 조합의 사진이 학습과 검증에 나뉘어 들어갈 수 있습니다.
    - 조합 단위로 나누면, 이 17개 클래스는 사진 3장이 통째로 학습이나 검증 한쪽에만 들어갑니다.
    - 같은 조합의 사진이 얼마나 비슷한지는 전체를 확인하지 않았습니다. 5-3(`K-003351-016262-018357`)과 5-7(`K-003351-033880-035206`)에서 본 두 조합은 같은 알약들이 비슷한 배치로 찍혀 있었습니다.
- 분할 방법은 팀원과 상의해 정합니다.

## 6-4. 정리

### 확인한 것

| 단계 | 검사 | 결과 |
|---|---|---|
| 6-1 | `category_id`와 약 이름의 짝 | 763개 모두 `category_id`, `dl_idx`, `categories`의 `id`가 같고, 약 이름도 같습니다. `category_id` 56개와 약 이름 56개가 1:1로 대응합니다. |
| 6-2 | 클래스별 bbox 수 | 클래스 56개, bbox 763개입니다. 클래스별 bbox 수는 최소 3개, 중앙값 9개, 최대 153개이고, 한 사진에 같은 클래스가 두 번 라벨된 경우는 없습니다. |
| 6-3 | 클래스별 조합 수 | 조합은 114개입니다. bbox가 3개뿐인 클래스 17개는 모두 한 조합의 사진 3장에만 나옵니다. |

### 클래스 분포의 특징

- 3351(일양하이트린정 2mg)이 bbox 153개로 전체의 20.1%이고, train 사진 232장 중 153장(약 66%)에 들어 있습니다. 그다음은 3483(45개), 35206(37개)입니다.
- bbox가 3개뿐인 클래스가 17개로, 전체 클래스의 약 30%입니다.
- 개수는 어노테이션에 적힌 그대로 센 값입니다. 4번의 라벨 누락을 반영하면 3351은 4개, 20014·32310·33880·4543은 1개씩 더 많아야 합니다.

### 정해야 할 것 (팀원과 상의)

- 클래스 목록(`classes.py`): 6-1 결과에 따라 `category_id` 56개와 약 이름을 1:1로 짝지어 만들 수 있습니다.
- 클래스 불균형 대응: 3351 하나가 bbox의 약 20%이고, bbox가 3개뿐인 클래스가 17개입니다.
- 학습/검증 분할: 사진 단위로 나누면 같은 조합의 사진이 학습과 검증에 나뉠 수 있고, 조합 단위로 나누면 bbox가 3개뿐인 클래스 17개가 통째로 한쪽에만 들어갑니다.
- 대회 설명대로 test에는 train에 없는 클래스가 있습니다. test에 라벨이 없어서 EDA로는 그 수를 확인할 수 없습니다.